# H286: begrenzter Gruppenversuch

**A100 auswählen → Alle ausführen.** Der kleine Starter lädt das bereitgestellte
private Paket, ohne das große Notebook im Browser zu öffnen. Kein Drive-Zugriff,
keine GPU-Warteschleife und keine weiteren manuellen Zwischenstufen.

Das originale Qwen3-1.7B, FP32 mit ausgeschaltetem TF32, dieselben 286 Koordinaten
und die feste REF-W-Normierung bleiben erhalten. Gruppenmessungen, begrenzte
Antwortläufe und Abschlussprüfungen laufen automatisch nacheinander.
Geplant sind zehn Eingaben, 350 kurze Prefills und 46 freie
Antwortläufe mit höchstens 128 ausgewählten Tokens. Fortschritt und tatsächliche
Zeiten werden protokolliert; Laden und technische Prüfungen kommen hinzu.

Die H286-Gruppen stammen aus der bestehenden Auswahl; dieser Versuch ist explorativ.
Er prüft keine allgemeine Leistungsverbesserung, kein Lernen und kein Gedächtnis.
Inhalt und verlangtes Antwortformat müssen später getrennt beurteilt werden.

Am Ende wird das Ergebnis- oder Diagnose-ZIP angeboten. Ein Abbruch wird als Fehler
gekennzeichnet; bereits gespeicherte Einzelresultate bleiben erhalten. Alte Ergebnisse werden nicht verändert;
vorhandene Modellgewichte im Cache werden wiederverwendet. Das private Paket nicht
öffentlich hochladen. Das öffentliche Template enthält keine privaten Referenzbytes.


In [ ]:
# Private Nutzdatei: unverändert lassen und Alle ausführen.
# Öffentliches Template: optional vorhandene Originaldateien in diesem Verzeichnis.
REFERENCE_DIRECTORY = ""
RESUME_ZIP = ""  # Nur für einen vorhandenen Checkpoint von einer anderen Sitzung.
RUN_DIRECTORY = ""  # Leer: automatisch, wiederholbar in derselben Sitzung.
MIN_FREE_VRAM_GIB = 12.0


In [ ]:
# Eingebettete Quellen und optionale private Originalbytes; nicht manuell ändern.
TRANSFORMERS_VERSION = '5.22.2'
ACCELERATE_VERSION = '1.16.0'
REFERENCE_SHA256 = {'hn_set.json': '11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd', 'htq_policy.json': '160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c', 'refw_h286.json': '68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc', 'group_continuity.json': '111d7213f76374a459c38ca17f4701eb9228693708ee1657a7e25f987c65ddc1'}
REFERENCE_FILES_B64 = {}
MODULE_SOURCES = {'attention_probe/__init__.py': '"""Isolated, optional attention-pulse pilot; existing experiment code is unchanged."""\n', 'attention_probe/reference.py': '"""Bind a new attention pilot to the private, exact DSSE H_tq selection.\n\nThis module validates identities and provenance only. It imports no model runtime,\ncontains no private neuron coordinates, and does not map them to another model.\n\n    python -m attention_probe.reference --hn-set /private/hn_set.json \\\n        --policy /private/htq_policy.json --out /private/reference.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import Any\n\n\nHN_SET_SHA256 = "11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd"\nPOLICY_SHA256 = "160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c"\nMODEL_ID = "qwen3-1.7b"\nMODEL_REPO = "Qwen/Qwen3-1.7B"\nMODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"\nN_COORDINATES, N_LAYERS, MLP_WIDTH = 286, 28, 6144\nH_POLICY = "htq_v2"\n\n\nclass ReferenceError(ValueError):\n    """The supplied files do not establish the required DSSE reference."""\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelPins:\n    repo: str\n    revision: str\n    sha256: tuple[tuple[str, str], ...]\n    model_id: str = MODEL_ID\n    n_layers: int = N_LAYERS\n    mlp_width: int = MLP_WIDTH\n\n    def to_dict(self) -> dict[str, Any]:\n        return dict(model_id=self.model_id, repo=self.repo, revision=self.revision,\n                    sha256=dict(self.sha256), n_layers=self.n_layers, mlp_width=self.mlp_width)\n\n\n@dataclass(frozen=True, slots=True)\nclass Reference:\n    coordinates: tuple[tuple[int, int], ...]\n    hn_sha256: str\n    policy_sha256: str\n    model: ModelPins\n    h_policy: str = H_POLICY\n    post_hoc: bool = True\n    amendment: int = 2\n\n    def to_dict(self) -> dict[str, Any]:\n        """Return a fresh JSON binding; its coordinates remain private run data."""\n        return dict(schema="dsse_h286_reference_v1", coordinates=[list(x) for x in self.coordinates],\n                    coordinate_order="hn_set.H_op (identical to H_tq)",\n                    n_coordinates=len(self.coordinates),\n                    source_sha256={"hn_set.json": self.hn_sha256, "htq_policy.json": self.policy_sha256},\n                    model=self.model.to_dict(), h_policy=self.h_policy, post_hoc=self.post_hoc,\n                    amendment=self.amendment, confirmatory_eligible=False,\n                    scope="Neuron identities only; no CETT equivalence or cross-model coordinate mapping.")\n\n\ndef _digest(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise ReferenceError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    out = {}\n    for key, value in pairs:\n        if key in out:\n            raise ReferenceError("Duplicate JSON object key.")\n        out[key] = value\n    return out\n\n\ndef _reject_constant(value: str) -> Any:\n    raise ReferenceError("Non-finite JSON constants are not allowed.")\n\n\ndef _document(raw: bytes, label: str) -> dict[str, Any]:\n    try:\n        value = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise ReferenceError(f"{label} is not valid JSON.") from exc\n    if not isinstance(value, dict):\n        raise ReferenceError(f"{label} must be a JSON object.")\n    return value\n\n\ndef _coordinates(value: Any, label: str) -> tuple[tuple[int, int], ...]:\n    if not isinstance(value, list) or len(value) != N_COORDINATES:\n        raise ReferenceError(f"{label} must contain exactly {N_COORDINATES} coordinates.")\n    result = []\n    for i, coordinate in enumerate(value):\n        if not isinstance(coordinate, list) or len(coordinate) != 2 or any(type(x) is not int for x in coordinate):\n            raise ReferenceError(f"{label}[{i}] must be a pair of integer coordinates.")\n        layer, index = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= index < MLP_WIDTH):\n            raise ReferenceError(f"{label}[{i}] is outside the 28 by 6144 MLP architecture.")\n        result.append((layer, index))\n    if len(set(result)) != N_COORDINATES:\n        raise ReferenceError(f"{label} contains duplicate coordinates.")\n    return tuple(result)\n\n\ndef _policy(document: dict[str, Any], label: str) -> None:\n    if document.get("h_policy") != H_POLICY or document.get("post_hoc") is not True:\n        raise ReferenceError(f"{label} must identify the post-hoc htq_v2 policy.")\n    if type(document.get("amendment")) is not int or document["amendment"] != 2:\n        raise ReferenceError(f"{label} must identify amendment 2.")\n    if document.get("H_op_name") != "H_tq":\n        raise ReferenceError(f"{label} must use H_tq as the operative set.")\n\n\ndef _model(document: dict[str, Any], label: str) -> ModelPins:\n    provenance = document.get("provenance")\n    if not isinstance(provenance, dict) or not isinstance(provenance.get("models"), dict):\n        raise ReferenceError(f"{label} has no model provenance.")\n    for field, expected in (("h_policy", H_POLICY), ("post_hoc", True), ("amendment", 2), ("mock", False)):\n        if field in provenance and (type(provenance[field]) is not type(expected) or provenance[field] != expected):\n            raise ReferenceError(f"{label} has conflicting {field} provenance.")\n    model = provenance["models"].get(MODEL_ID)\n    if not isinstance(model, dict) or model.get("repo") != MODEL_REPO or model.get("revision") != MODEL_REVISION:\n        raise ReferenceError(f"{label} does not identify the pinned Qwen3-1.7B model.")\n    hashes = model.get("sha256")\n    if not isinstance(hashes, dict) or not hashes or any(not isinstance(k, str) or not k for k in hashes):\n        raise ReferenceError(f"{label} has no model-file SHA-256 pins.")\n    pins = tuple(sorted((name, _sha(digest, f"{label} model file")) for name, digest in hashes.items()))\n    return ModelPins(repo=model["repo"], revision=model["revision"], sha256=pins)\n\n\ndef verify_documents(hn_bytes: bytes, policy_bytes: bytes, *, expected_hn_sha256: str,\n                     expected_policy_sha256: str) -> Reference:\n    """Low-level verification with explicit trust anchors, including synthetic tests.\n\n    Normal pilot code must call load_reference(), whose anchors are the production\n    hashes. Checksums cover original bytes, so JSON reformatting is not equivalent.\n    """\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected hn_set hash")\n    expected_policy_sha256 = _sha(expected_policy_sha256, "Expected policy hash")\n    if not isinstance(hn_bytes, bytes) or not isinstance(policy_bytes, bytes):\n        raise ReferenceError("Reference documents must be immutable bytes.")\n    hn_digest, policy_digest = _digest(hn_bytes), _digest(policy_bytes)\n    if hn_digest != expected_hn_sha256:\n        raise ReferenceError("hn_set.json SHA-256 mismatch.")\n    if policy_digest != expected_policy_sha256:\n        raise ReferenceError("htq_policy.json SHA-256 mismatch.")\n    hn, policy = _document(hn_bytes, "hn_set.json"), _document(policy_bytes, "htq_policy.json")\n    _policy(hn, "hn_set.json"); _policy(policy, "htq_policy.json")\n    coordinates = _coordinates(hn.get("H_op"), "H_op")\n    if coordinates != _coordinates(hn.get("H_tq"), "H_tq"):\n        raise ReferenceError("H_op and H_tq must be identical, including coordinate order.")\n    if type(hn.get("n_H")) is not int or hn["n_H"] != N_COORDINATES:\n        raise ReferenceError("hn_set n_H must equal 286.")\n    if type(policy.get("n_H_op")) is not int or policy["n_H_op"] != N_COORDINATES:\n        raise ReferenceError("Policy n_H_op must equal 286.")\n    hn_model, policy_model = _model(hn, "hn_set.json"), _model(policy, "htq_policy.json")\n    if hn_model != policy_model:\n        raise ReferenceError("The two files have different model-file pins.")\n    return Reference(coordinates=coordinates, hn_sha256=hn_digest,\n                     policy_sha256=policy_digest, model=hn_model)\n\n\ndef load_reference(hn_set_path: str | Path, policy_path: str | Path) -> Reference:\n    """Load only the exact DSSE production files; no caller-supplied hash bypass."""\n    return verify_documents(Path(hn_set_path).read_bytes(), Path(policy_path).read_bytes(),\n                            expected_hn_sha256=HN_SET_SHA256, expected_policy_sha256=POLICY_SHA256)\n\n\ndef write_reference(reference: Reference, out: str | Path) -> None:\n    """Atomically write a binding to the caller\'s explicit private output path."""\n    path = Path(out)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(reference.to_dict(), indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,\n                                         prefix="." + path.name + ".", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            handle.write(payload); handle.flush(); os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--hn-set", required=True, type=Path)\n    parser.add_argument("--policy", required=True, type=Path)\n    parser.add_argument("--out", required=True, type=Path)\n    args = parser.parse_args(argv)\n    if args.out.resolve() in (args.hn_set.resolve(), args.policy.resolve()):\n        parser.error("--out must not overwrite a source document.")\n    try:\n        reference = load_reference(args.hn_set, args.policy)\n        write_reference(reference, args.out)\n    except (OSError, ReferenceError) as exc:\n        parser.error(str(exc))\n    print(f"Validated {len(reference.coordinates)} H_tq coordinates; private binding written to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'attention_probe/fixed_refw.py': '"""Validate the private, fixed H286 REF-W extract without model imports.\n\nThe compact file contains existing population means/SDs, already cast to FP32\nas in DSSE\'s SetReader. This module neither fits nor recomputes normalization.\nCETT uses max(||down_proj(z)||, CETT_EPS); zCETT uses the saved c_mu and\nmax(c_sd, STANDARDIZATION_EPS). Zero saved SD is valid and stays zero here:\nthe consuming measurement applies the documented denominator floor.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport struct\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .reference import (\n    HN_SET_SHA256,\n    MLP_WIDTH,\n    N_COORDINATES,\n    N_LAYERS,\n    POLICY_SHA256,\n    Reference,\n    ReferenceError,\n)\n\nFIXED_REFW_SHA256 = "68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc"\nSOURCE_HN_META_SHA256 = "9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff"\nCETT_EPS = 1e-9\nSTANDARDIZATION_EPS = 1e-6\nSCHEMA = "dsse_h286_refw_v1"\n_VECTOR_NAMES = {"z_mu", "z_sd", "c_mu", "c_sd"}\n_FIELDS = {"schema", "source_hn_meta_sha256", "hn_sha256", "coordinates", "refw"}\n\n\nclass FixedRefWError(ReferenceError):\n    """The supplied extract does not establish the fixed normalization."""\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise FixedRefWError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise FixedRefWError("Duplicate JSON object key.")\n        result[key] = value\n    return result\n\n\ndef _reject_constant(value: str) -> Any:\n    raise FixedRefWError(f"Non-finite JSON constant: {value}")\n\n\ndef verify_fixed_refw(\n    raw: bytes,\n    reference: Reference,\n    *,\n    expected_sha256: str,\n    expected_source_hn_meta_sha256: str,\n    expected_hn_sha256: str,\n) -> dict[str, Any]:\n    """Low-level validation with explicit trust anchors for synthetic tests.\n\n    Production callers must use load_fixed_refw(), which fixes all three\n    anchors. The source hash binds the previously verified full hn_meta file;\n    this compact loader does not claim to inspect that large file again.\n    """\n    expected_sha256 = _sha(expected_sha256, "Expected compact hash")\n    expected_source_hn_meta_sha256 = _sha(expected_source_hn_meta_sha256, "Expected source hash")\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected H selection hash")\n    if not isinstance(raw, bytes):\n        raise FixedRefWError("The compact document must be immutable bytes.")\n    if hashlib.sha256(raw).hexdigest() != expected_sha256:\n        raise FixedRefWError("Fixed REF-W SHA-256 mismatch.")\n    if not isinstance(reference, Reference) or reference.hn_sha256 != expected_hn_sha256:\n        raise FixedRefWError("Reference has a different H selection hash.")\n    try:\n        document = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise FixedRefWError("Fixed REF-W is not valid JSON.") from exc\n    if not isinstance(document, dict) or set(document) != _FIELDS:\n        raise FixedRefWError("Unexpected fixed REF-W document fields.")\n    if document["schema"] != SCHEMA:\n        raise FixedRefWError("Unsupported fixed REF-W schema.")\n    if document["source_hn_meta_sha256"] != expected_source_hn_meta_sha256:\n        raise FixedRefWError("Full hn_meta source hash mismatch.")\n    if document["hn_sha256"] != expected_hn_sha256:\n        raise FixedRefWError("Compact H selection hash mismatch.")\n\n    coordinates = document["coordinates"]\n    if not isinstance(coordinates, list) or len(coordinates) != N_COORDINATES:\n        raise FixedRefWError("Expected exactly 286 ordered coordinates.")\n    checked = []\n    for coordinate in coordinates:\n        if (not isinstance(coordinate, list) or len(coordinate) != 2\n                or any(type(value) is not int for value in coordinate)):\n            raise FixedRefWError("Coordinates must be integer pairs.")\n        layer, neuron = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= neuron < MLP_WIDTH):\n            raise FixedRefWError("Coordinate outside the Qwen3-1.7B MLP architecture.")\n        checked.append((layer, neuron))\n    if len(set(checked)) != N_COORDINATES or tuple(checked) != reference.coordinates:\n        raise FixedRefWError("Coordinates differ from the ordered H286 reference.")\n\n    vectors = document["refw"]\n    if not isinstance(vectors, dict) or set(vectors) != _VECTOR_NAMES:\n        raise FixedRefWError("Expected exactly z_mu, z_sd, c_mu and c_sd.")\n    for name, vector in vectors.items():\n        if not isinstance(vector, list) or len(vector) != N_COORDINATES:\n            raise FixedRefWError(f"{name} must have exactly 286 values.")\n        for value in vector:\n            if type(value) not in (int, float):\n                raise FixedRefWError(f"{name} must contain finite FP32 numbers.")\n            try:\n                fp32 = struct.unpack("<f", struct.pack("<f", value))[0]\n            except (OverflowError, struct.error) as exc:\n                raise FixedRefWError(f"{name} contains a value outside FP32.") from exc\n            if not math.isfinite(fp32) or fp32 != value:\n                raise FixedRefWError(f"{name} contains a non-finite or non-FP32 value.")\n            if name.endswith("_sd") and value < 0:\n                raise FixedRefWError(f"{name} cannot contain negative standard deviations.")\n    return document\n\n\ndef load_fixed_refw(path: str | Path, reference: Reference) -> dict[str, Any]:\n    """Load only the exact production extract bound to the original reference."""\n    if not isinstance(reference, Reference) or reference.policy_sha256 != POLICY_SHA256:\n        raise FixedRefWError("Expected the original production reference policy.")\n    return verify_fixed_refw(\n        Path(path).read_bytes(), reference,\n        expected_sha256=FIXED_REFW_SHA256,\n        expected_source_hn_meta_sha256=SOURCE_HN_META_SHA256,\n        expected_hn_sha256=HN_SET_SHA256,\n    )\n', 'attention_probe/interventions.py': '"""One-pass attention pulses and read-only activation capture for Qwen3.\n\nPositions are zero-based columns of the complete input sequence, not relative\nindices of a cached decode chunk. Selected positions must be real tokens in\nevery batch row. Qwen3 queries are scaled *after* q_norm and before RoPE, so\nnormalization cannot cancel the pulse. Scaling queries scales their attention\nlogits; it makes no claim about FLOPs, energy, or a computational bottleneck.\n\nCoordinateCapture stores raw gated-MLP z at down_proj input, not CETT.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nimport math\nimport inspect\nfrom numbers import Integral\nfrom typing import Iterable, Sequence\n\nimport torch\n\n\ndef _integers(values: Iterable[int], name: str, *, nonempty: bool = True) -> tuple[int, ...]:\n    result = tuple(values)\n    if nonempty and not result:\n        raise ValueError(f"{name} must not be empty")\n    if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in result):\n        raise ValueError(f"{name} must contain nonnegative integers")\n    if len(set(result)) != len(result):\n        raise ValueError(f"{name} must not contain duplicates")\n    return tuple(int(x) for x in result)\n\n\ndef _layers(model):\n    base = getattr(model, "model", model)\n    layers = getattr(base, "layers", None)\n    if layers is None or not len(layers):\n        raise ValueError("Expected Hugging Face decoder model.model.layers (or model.layers)")\n    return layers\n\n\ndef _cache_length(cache) -> int | None:\n    if cache is None:\n        return 0\n    getter = getattr(cache, "get_seq_length", None)\n    if getter is not None:\n        try:\n            return int(getter())\n        except (TypeError, ValueError, RuntimeError):\n            return None\n    if isinstance(cache, (tuple, list)):\n        if not cache:\n            return 0\n        try:\n            return int(cache[0][0].shape[-2])\n        except (IndexError, AttributeError, TypeError):\n            return None\n    return None\n\n\nclass _OneFullPass:\n    """Bind hooks to one complete, uncached outer model forward and clean them up."""\n\n    def __init__(self, model, positions: Sequence[int]):\n        self.model = model\n        self.positions = _integers(positions, "positions")\n        self._handles = []\n        self._entered = False\n        self._used = False\n        self._claimed = False\n        self._window = None\n        self.full_passes = 0\n        self.skipped_cached_passes = 0\n\n    def _model_pre(self, module, args, kwargs):\n        self._window = None\n        if self._claimed:\n            return\n        # HF forwards allow both positional and keyword cache arguments.\n        try:\n            named = inspect.signature(module.forward).bind_partial(*args, **kwargs).arguments\n        except (TypeError, ValueError):\n            named = kwargs\n        fields = dict(kwargs)\n        fields.update({name: value for name, value in named.items() if name not in ("args", "kwargs")})\n        cache = fields.get("past_key_values", fields.get("past_key_value"))\n        length = _cache_length(cache)\n        cache_position = fields.get("cache_position")\n        nonzero_offset = (cache_position is not None and torch.as_tensor(cache_position).numel()\n                          and int(torch.as_tensor(cache_position).reshape(-1)[0]) != 0)\n        if length is None or length > 0 or nonzero_offset:\n            self.skipped_cached_passes += 1\n            return\n        ids = fields.get("input_ids")\n        embeds = fields.get("inputs_embeds")\n        value = ids if ids is not None else embeds\n        if value is None and args:\n            value = args[0]\n        if not isinstance(value, torch.Tensor) or value.ndim not in (2, 3):\n            raise ValueError("Cannot identify full input sequence for absolute token positions")\n        batch, tokens = value.shape[:2]\n        if max(self.positions) >= tokens:\n            raise ValueError(f"position {max(self.positions)} is outside the full sequence of {tokens} tokens")\n        mask = fields.get("attention_mask")\n        if mask is not None:\n            if tuple(mask.shape) != (batch, tokens):\n                raise ValueError("Expected a two-dimensional outer attention mask")\n            if not bool(mask[:, list(self.positions)].bool().all()):\n                raise ValueError("Selected absolute position is padding in at least one batch row")\n        self._claimed = True\n        self.full_passes += 1\n        self._window = (int(batch), int(tokens))\n\n    def _model_post(self, module, args, kwargs, output):\n        self._window = None\n\n    def _install(self):\n        raise NotImplementedError\n\n    def validate_fired(self):\n        raise NotImplementedError\n\n    def __enter__(self):\n        if self._used:\n            raise RuntimeError("Use a fresh hook context for each attempt")\n        if self.model.training:\n            raise ValueError("Attention pilot requires model.eval(); model mode is never changed by hooks")\n        self._used = self._entered = True\n        try:\n            self._handles.append(self.model.register_forward_pre_hook(self._model_pre, with_kwargs=True))\n            self._handles.append(self.model.register_forward_hook(self._model_post, with_kwargs=True, always_call=True))\n            self._install()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def close(self):\n        for handle in reversed(self._handles):\n            handle.remove()\n        self._handles.clear()\n        self._window = None\n        self._entered = False\n\n    def __exit__(self, kind, error, tb):\n        self.close()\n        if kind is None:\n            self.validate_fired()\n        return False\n\n\nclass AttentionPulse(_OneFullPass):\n    """Scale post-q_norm queries, or add a per-head matched random delta, once.\n\n    Primary architecture is config.model_type=\'qwen3\'. Qwen2\'s unnormalized\n    q_proj output is supported only with explicit architecture=\'qwen2\'. Random\n    deltas match ||(beta-1) q|| independently for each selected token/query head\n    before output-dtype rounding. actual_delta_norms reports rounding effects.\n    A local CPU generator never reads or changes PyTorch\'s global RNG state.\n    """\n\n    def __init__(self, model, block: int, positions: Sequence[int], beta: float = 1.0,\n                 mode: str = "scale", seed: int = 0, *, architecture: str = "qwen3"):\n        super().__init__(model, positions)\n        if not isinstance(beta, (int, float)) or not math.isfinite(beta) or not 0 <= beta <= 1:\n            raise ValueError("beta must be finite and between zero and one")\n        if mode not in ("scale", "random"):\n            raise ValueError("mode must be \'scale\' or \'random\'")\n        if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n        self.layers = _layers(model)\n        self.block = _integers((block,), "block")[0]\n        if self.block >= len(self.layers):\n            raise ValueError("block is outside the decoder")\n        if architecture not in ("qwen3", "qwen2") or getattr(model.config, "model_type", None) != architecture:\n            raise ValueError("Unsupported architecture: Qwen3 is primary; Qwen2 requires explicit opt-in")\n        self.architecture, self.mode, self.beta = architecture, mode, float(beta)\n        self.seed = int(seed)\n        attention = self.layers[self.block].self_attn\n        self.heads = int(model.config.num_attention_heads)\n        self.head_dim = int(getattr(attention, "head_dim", getattr(model.config, "head_dim", 0)))\n        if self.heads <= 0 or self.head_dim <= 0:\n            raise ValueError("Query-head shape is not declared")\n        if architecture == "qwen3":\n            self.hook_module = getattr(attention, "q_norm", None)\n            self.hook_location = "self_attn.q_norm.output (B,T,H,D), before RoPE"\n            if self.hook_module is None:\n                raise ValueError("Qwen3 requires post-q_norm intervention; q_proj fallback is forbidden")\n        else:\n            if getattr(attention, "q_norm", None) is not None:\n                raise ValueError("Qwen2 fallback expects no query normalization")\n            self.hook_module = getattr(attention, "q_proj", None)\n            self.hook_location = "self_attn.q_proj.output, explicit Qwen2 fallback"\n            if self.hook_module is None:\n                raise ValueError("Qwen2 query projection is absent")\n        self.fired_count = 0\n        self.requested_delta_norms = None\n        self.actual_delta_norms = None\n        self._generator = torch.Generator(device="cpu").manual_seed(self.seed)\n\n    def _install(self):\n        self._handles.append(self.hook_module.register_forward_hook(self._pulse))\n\n    def _pulse(self, module, args, output):\n        if self._window is None or self.fired_count:\n            return None\n        batch, tokens = self._window\n        if not isinstance(output, torch.Tensor):\n            raise ValueError("Query hook did not return a tensor")\n        shape = (batch, tokens, self.heads, self.head_dim)\n        if self.architecture == "qwen3":\n            if tuple(output.shape) != shape:\n                raise ValueError(f"Unsupported Qwen3 post-q_norm layout: expected {shape}, got {tuple(output.shape)}")\n            query = output\n        else:\n            if tuple(output.shape) != (batch, tokens, self.heads * self.head_dim):\n                raise ValueError("Unsupported Qwen2 q_proj output shape")\n            query = output.reshape(shape)\n        selected = query[:, list(self.positions), :, :]\n        if not bool(torch.isfinite(selected).all()):\n            raise ValueError("Nonfinite query in selected pulse positions")\n        self.fired_count += 1\n        norms = (1-self.beta) * selected.detach().double().cpu().norm(dim=-1)\n        self.requested_delta_norms = norms.clone()\n        if self.beta == 1:\n            self.actual_delta_norms = torch.zeros_like(norms)\n            return None  # Exact sham: preserve the original tensor, no multiplication or RNG draw.\n        if self.mode == "scale":\n            changed = selected * self.beta\n        else:\n            random = torch.randn(selected.shape, generator=self._generator, dtype=torch.float64, device="cpu")\n            random = random / random.norm(dim=-1, keepdim=True).clamp_min(torch.finfo(torch.float64).tiny)\n            delta = (random * norms[..., None]).to(device=selected.device, dtype=selected.dtype)\n            changed = selected + delta\n        self.actual_delta_norms = (changed.detach().double().cpu()-selected.detach().double().cpu()).norm(dim=-1)\n        result = query.clone()\n        result[:, list(self.positions), :, :] = changed\n        return result if self.architecture == "qwen3" else result.reshape(output.shape)\n\n    def validate_fired(self):\n        if self.fired_count != 1:\n            raise RuntimeError(f"Attention pulse must fire once on a full/prefill pass; fired {self.fired_count}")\n        return self.fired_count\n\n\nclass CoordinateCapture(_OneFullPass):\n    """Read exact ordered down_proj-input coordinates and decoder residuals.\n\n    values is CPU float32 [batch, positions, coordinates]; coordinate order is\n    exactly the supplied order. residuals maps each requested decoder layer to\n    CPU float32 [batch, positions, hidden_size]. No input/output tensor changes.\n    This is raw z capture, not CETT or a causal-neuron importance estimate.\n    """\n\n    feature_kind = "raw gated-MLP activation z at down_proj input; not CETT"\n\n    def __init__(self, model, coordinates: Sequence[tuple[int, int]], positions: Sequence[int],\n                 residual_layers: Sequence[int] | None = None, *, expected_count: int = 286):\n        super().__init__(model, positions)\n        self.layers = _layers(model)\n        if getattr(model.config, "model_type", None) != "qwen3":\n            raise ValueError("Original H coordinates require the Qwen3 decoder architecture")\n        if isinstance(expected_count, bool) or not isinstance(expected_count, Integral) or expected_count < 1:\n            raise ValueError("expected_count must be a positive integer")\n        self.coordinates = tuple(tuple(pair) for pair in coordinates)\n        if len(self.coordinates) != expected_count or not self.coordinates:\n            raise ValueError(f"Expected exactly {expected_count} ordered H coordinates")\n        if len(set(self.coordinates)) != len(self.coordinates):\n            raise ValueError("Duplicate H coordinates")\n        self._by_layer = defaultdict(list)\n        for output_index, pair in enumerate(self.coordinates):\n            if len(pair) != 2:\n                raise ValueError("Each coordinate must be (block, down_proj_input_index)")\n            # A coordinate may have equal block/channel numbers; duplicates are\n            # forbidden across coordinate pairs, not within a pair.\n            if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in pair):\n                raise ValueError("Coordinate indices must be nonnegative integers")\n            block, neuron = (int(x) for x in pair)\n            if block >= len(self.layers):\n                raise ValueError("Coordinate block is outside the decoder")\n            down = getattr(getattr(self.layers[block], "mlp", None), "down_proj", None)\n            if down is None or not hasattr(down, "in_features") or neuron >= down.in_features:\n                raise ValueError("Coordinate channel is outside mlp.down_proj input")\n            self._by_layer[block].append((output_index, neuron))\n        self.residual_layers = _integers(residual_layers if residual_layers is not None else sorted(self._by_layer),\n                                        "residual_layers", nonempty=False)\n        if any(block >= len(self.layers) for block in self.residual_layers):\n            raise ValueError("Residual layer is outside the decoder")\n        self._values = None\n        self.residuals = {}\n        self._seen = set()\n        self.fired_count = 0\n\n    def _install(self):\n        for block in self._by_layer:\n            self._handles.append(self.layers[block].mlp.down_proj.register_forward_pre_hook(self._coordinate_hook(block)))\n        for block in self.residual_layers:\n            self._handles.append(self.layers[block].register_forward_hook(self._residual_hook(block)))\n\n    def _coordinate_hook(self, block):\n        def hook(module, args):\n            if self._window is None:\n                return None\n            if block in self._seen:\n                raise RuntimeError("down_proj was called twice in the selected full pass")\n            z = args[0]\n            batch, tokens = self._window\n            if not isinstance(z, torch.Tensor) or z.ndim != 3 or tuple(z.shape[:2]) != (batch, tokens):\n                raise ValueError("Unsupported down_proj input shape")\n            pairs = self._by_layer[block]\n            observed = z[:, list(self.positions), :][:, :, [channel for _, channel in pairs]].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected raw activation")\n            if self._values is None:\n                self._values = torch.empty((batch, len(self.positions), len(self.coordinates)), dtype=torch.float32)\n            self._values[:, :, [index for index, _ in pairs]] = observed\n            self._seen.add(block)\n            self.fired_count = int(self._seen == set(self._by_layer))\n            return None\n        return hook\n\n    def _residual_hook(self, block):\n        def hook(module, args, output):\n            if self._window is None:\n                return None\n            if block in self.residuals:\n                raise RuntimeError("Decoder layer was called twice in the selected full pass")\n            value = output[0] if isinstance(output, (tuple, list)) else output\n            if not isinstance(value, torch.Tensor) or value.ndim != 3 or tuple(value.shape[:2]) != self._window:\n                raise ValueError("Unsupported decoder residual output shape")\n            observed = value[:, list(self.positions)].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected decoder residual")\n            self.residuals[block] = observed\n            return None\n        return hook\n\n    @property\n    def values(self):\n        self.validate_fired()\n        return self._values\n\n    def validate_fired(self):\n        if self.fired_count != 1 or self._seen != set(self._by_layer) or set(self.residuals) != set(self.residual_layers):\n            raise RuntimeError("Raw-coordinate capture did not complete exactly one full/prefill pass")\n        return self.fired_count\n', 'attention_probe/pilot_items.json': '[\n  {\n    "id": "p01_correct_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 15. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p01_wrong_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 14. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p02_correct_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 5. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p02_wrong_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 6. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p03_correct_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 18. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p03_wrong_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 19. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p04_correct_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 4. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p04_wrong_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 3. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p05_correct_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: blue. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p05_wrong_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: pink. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p06_correct_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: pink. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p06_wrong_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: blue. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p07_correct_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: green. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p07_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: brown. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p08_correct_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: B. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  },\n  {\n    "id": "p08_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: A. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  }\n]\n', 'qwen3_h286_runtime.py': '"""Short exploratory ORIGINAL Qwen3/H286 pilot; embedded notebook entry point.\n\nNo re-selection, fitting, generation, training, or cross-model coordinate mapping.\nOriginal FP32, TF32-off, fixed REF-W and efficient SDPA are retained. The two\ntechnical checks precede a fixed 16-request/five-arm exploratory run. Results do\nnot constitute a held-out confirmatory test or a proven attention bottleneck.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nMODEL = \'Qwen/Qwen3-1.7B\'\nREVISION = \'70d244cc86ccca08cf5af4e1e306ecf908b1ad5e\'\nTRANSFORMERS = \'5.22.2\'\nACCELERATE = \'1.16.0\'\nREFW_SHA256 = \'68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc\'\nMETA_SHA256 = \'9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff\'\nSEED = 20261117\nARMS = (\n    {\'name\': \'sham\', \'block\': 14, \'beta\': 1., \'mode\': \'scale\'},\n    {\'name\': \'soft_103\', \'block\': 14, \'beta\': 1 / 1.03, \'mode\': \'scale\'},\n    {\'name\': \'soft_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n    {\'name\': \'random_matched_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'random\'},\n    {\'name\': \'other_block_108\', \'block\': 7, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n)\n# Declared before any new model result. These are technical/exploratory items,\n# not the frozen DSSE evaluation sample or a benchmark accuracy test.\nITEMS_SHA256 = \'12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3\'\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False, allow_nan=False).encode()\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 << 20), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=\'.\' + path.name, delete=False) as handle:\n        temporary = Path(handle.name)\n        handle.write(canonical(value) + b\'\\n\')\n        handle.flush()\n        os.fsync(handle.fileno())\n    try:\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef archive(output):\n    target = output / \'qwen3_h286_results.zip\'\n    temporary = output / \'.qwen3_h286_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for item in sorted(output.rglob(\'*\')):\n            if item.is_file() and item not in (target, temporary) and not any(part.startswith(\'.\') for part in item.relative_to(output).parts):\n                bundle.write(item, item.relative_to(output))\n    os.replace(temporary, target)\n    return target\n\n\ndef load_inputs(hn_path, policy_path, refw_path, items_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn_path, policy_path)\n    stats = load_fixed_refw(refw_path, reference)\n    raw = Path(items_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ITEMS_SHA256:\n        raise ValueError(\'The exact prespecified 16-item file is required; no replacement or post-result selection\')\n    items = json.loads(raw)\n    if len(items) != 16 or len({item[\'id\'] for item in items}) != 16:\n        raise ValueError(\'Exactly 16 unique fixed requests are required\')\n    return reference, stats, items\n\n\ndef hardware_preflight(torch, cache_dir):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU erforderlich; keine Warteschleife. A100 empfohlen, L4 ausreichend.\')\n    free, total = torch.cuda.mem_get_info(0)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    disk = shutil.disk_usage(cache_dir).free\n    if free < 10 * (1 << 30):\n        raise RuntimeError(\'Mindestens 10 GiB freie GPU-VRAM für unverändertes FP32 erforderlich.\')\n    if disk < 8 * (1 << 30):\n        raise RuntimeError(\'Mindestens 8 GiB freier Cache-Speicher erforderlich; keine Gewichte heruntergeladen.\')\n    available = None\n    meminfo = Path(\'/proc/meminfo\')\n    if meminfo.exists():\n        fields = dict(line.split(\':\', 1) for line in meminfo.read_text().splitlines())\n        available = int(fields[\'MemAvailable\'].split()[0]) * 1024\n        if available < 10 * (1 << 30):\n            raise RuntimeError(\'Mindestens 10 GiB verfügbarer Host-RAM erforderlich.\')\n    return {\'gpu\': torch.cuda.get_device_name(0), \'free_vram\': free, \'total_vram\': total,\n            \'free_disk\': disk, \'available_host_ram\': available, \'device\': \'cuda:0\'}\n\n\ndef model_load_kwargs(torch):\n    return {\'revision\': REVISION, \'dtype\': torch.float32, \'device_map\': {\'\': 0},\n            \'attn_implementation\': \'sdpa\', \'trust_remote_code\': False, \'use_safetensors\': True,\n            \'local_files_only\': True}\n\n\ndef load_model(reference, cache_dir):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n    snapshot = Path(snapshot_download(MODEL, revision=REVISION, cache_dir=str(cache_dir),\n                                     allow_patterns=[\'*.json\', \'*.safetensors\', \'*.jinja\']))\n    if snapshot.name != REVISION:\n        raise ValueError(\'HF snapshot path does not identify the exact pinned revision\')\n    for filename, expected in reference.model.sha256:\n        if sha_file(snapshot / filename) != expected:\n            raise ValueError(\'Original model/tokenizer SHA mismatch: \' + filename)\n    config = AutoConfig.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                       local_files_only=True)\n    if config.model_type != \'qwen3\' or config.num_hidden_layers != 28 or config.intermediate_size != 6144:\n        raise ValueError(\'Original Qwen3-1.7B architecture mismatch\')\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                             local_files_only=True)\n    tokenizer.padding_side = \'left\'\n    model = AutoModelForCausalLM.from_pretrained(str(snapshot), config=config, **model_load_kwargs(torch))\n    model.eval().requires_grad_(False)\n    if any(parameter.device.type != \'cuda\' or parameter.device.index != 0 or parameter.dtype != torch.float32\n           for parameter in model.parameters()):\n        raise ValueError(\'All model parameters must remain full FP32 on cuda:0; no offload/quantization\')\n    return tokenizer, model\n\n\ndef prepare_request(tokenizer, index, item):\n    import torch\n    category, question, answer = item[\'category\'], item[\'prompt\'], item[\'target\']\n    formatted = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': question}], tokenize=False,\n                                              add_generation_prompt=True, enable_thinking=False)\n    prompt = tokenizer(formatted, add_special_tokens=False)[\'input_ids\']\n    answer_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not prompt or not answer_ids or len(prompt) + len(answer_ids) > 128:\n        raise ValueError(\'Unexpected empty or long fixed teacher-forcing request; no silent truncation\')\n    # Token at p predicts token p+1. The last answer activation is recorded but\n    # cannot cause its own prediction. Answer likelihood uses preceding positions.\n    ids = prompt + answer_ids\n    positions = list(range(max(0, len(ids) - 4), len(ids)))\n    return {\'request_id\': f\'r{index:03d}\', \'item_id\': item[\'id\'], \'pair_id\': item[\'pair_id\'], \'category\': category, \'question\': question, \'answer\': answer,\n            \'formatted_prompt\': formatted, \'formatted_prompt_sha256\': hashlib.sha256(formatted.encode()).hexdigest(), \'prompt_ids\': prompt, \'answer_ids\': answer_ids,\n            \'input_ids\': ids, \'capture_positions\': positions, \'pulse_positions\': [len(prompt) - 1],\n            \'answer_logit_positions\': list(range(len(prompt) - 1, len(ids) - 1)),\n            \'thinking_enabled\': False, \'ids\': torch.tensor([ids], dtype=torch.long)}\n\n\n@contextlib.contextmanager\ndef cett_tap(model, coordinates, positions, weight_norms=None):\n    import torch\n    groups = defaultdict(list)\n    for order, (layer, neuron) in enumerate(coordinates):\n        groups[layer].append((order, neuron))\n    result = {\'cett\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32),\n              \'denominator\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32)}\n    handles, seen = [], set()\n    try:\n        for layer, pairs in groups.items():\n            down = model.model.layers[layer].mlp.down_proj\n            columns = [neuron for _, neuron in pairs]\n            weight_norm = (weight_norms[layer] if weight_norms is not None\n                           else down.weight.float().norm(dim=0)[columns])\n\n            def hook(_module, args, output, layer=layer, pairs=pairs, columns=columns, weight_norm=weight_norm):\n                if layer in seen:\n                    raise ValueError(\'CETT tap fired repeatedly\')\n                z = args[0][:, positions][:, :, columns].abs().float()\n                den = output.norm(dim=-1)[:, positions].float()\n                contribution = z * weight_norm / den.clamp_min(1e-9)[..., None]\n                order = [index for index, _ in pairs]\n                result[\'cett\'][:, :, order] = contribution.detach().cpu()\n                result[\'denominator\'][:, :, order] = den[..., None].expand_as(contribution).detach().cpu()\n                seen.add(layer)\n            handles.append(down.register_forward_hook(hook))\n        yield result\n        if seen != set(groups):\n            raise ValueError(\'CETT capture incomplete\')\n    finally:\n        for handle in reversed(handles):\n            handle.remove()\n\n\ndef parameter_evidence(model):\n    return tuple((name, id(value), value.data_ptr(), value._version, str(value.dtype), str(value.device))\n                 for name, value in model.named_parameters())\n\n\ndef run_arm(model, request, coordinates, stats, arm, *, efficient_gpu=True, include_logits=False, weight_norms=None):\n    import torch\n\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    device = next(model.parameters()).device\n    ids = request[\'ids\'].to(device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    started = time.monotonic()\n    with (torch.inference_mode(), attention,\n          CoordinateCapture(model, coordinates, request[\'capture_positions\'],\n                            residual_layers=tuple(layer for layer in (3, 7, 14, 27) if layer < len(model.model.layers)),\n                            expected_count=len(coordinates)) as capture,\n          cett_tap(model, coordinates, request[\'capture_positions\'], weight_norms) as contributions,\n          AttentionPulse(model, arm[\'block\'], request[\'pulse_positions\'], arm[\'beta\'], arm[\'mode\'],\n                         seed=SEED + int(request[\'request_id\'][1:])) as pulse):\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n        if not bool(torch.isfinite(output.logits).all()):\n            raise ValueError(\'Nonfinite model logits\')\n        selected_logits = output.logits[:, request[\'answer_logit_positions\']].float()\n        targets = torch.tensor(request[\'answer_ids\'], device=device)[None, :, None]\n        logprob = selected_logits.log_softmax(-1).gather(-1, targets).squeeze(-1).cpu()\n        raw = capture.values\n        cett = contributions[\'cett\']\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        zcett = (cett - mu) / sd\n        result = {\'arm\': arm, \'answer_token_logprobs\': logprob.flatten().tolist(),\n                  \'answer_logprob_sum\': float(logprob.sum()), \'answer_logprob_mean\': float(logprob.mean()),\n                  \'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count,\n                  \'requested_delta_norms\': pulse.requested_delta_norms.tolist(),\n                  \'actual_delta_norms\': pulse.actual_delta_norms.tolist(),\n                  \'elapsed_seconds\': time.monotonic() - started}\n        arrays = {\'raw_z\': raw.numpy(), \'cett\': cett.numpy(), \'zcett\': zcett.numpy(),\n                  \'down_output_norm_raw\': contributions[\'denominator\'].numpy(),\n                  \'gold_token_logprob\': logprob.numpy()}\n        for layer, value in capture.residuals.items():\n            arrays[f\'residual_layer_{layer}\'] = value.numpy()\n        if include_logits:\n            arrays[\'technical_logits\'] = selected_logits.detach().cpu().numpy()\n    return result, arrays\n\n\ndef plan_document(reference, stats, items):\n    import attention_probe.reference as reference_module\n    module_root = Path(reference_module.__file__).parent\n    module_hashes = {name: sha_file(module_root / name) for name in (\'reference.py\', \'interventions.py\', \'fixed_refw.py\')}\n    return {\'schema\': \'qwen3_h286_short_pilot_v1\', \'model\': reference.model.to_dict(),\n            \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n            \'refw_sha256\': REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n            \'coordinates\': [list(pair) for pair in reference.coordinates], \'seed\': SEED,\n            \'arms\': list(ARMS), \'items\': items, \'items_sha256\': ITEMS_SHA256,\n            \'helper_sha256\': sha_file(__file__), \'module_sha256\': module_hashes,\n            \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'sdpa/EFFICIENT_ATTENTION\',\n            \'transformers\': TRANSFORMERS, \'thinking_enabled\': False,\n            \'claim\': \'externally supplied draft-checking proxy; not own prior output or proof of self-monitoring; no held-out confirmatory test\',\n            \'candidate_block\': \'14 zero-based, prespecified candidate; not a proven bottleneck\',\n            \'cett\': \'abs(z)*norm(Wdown_col)/max(norm(down(z)),1e-9)\',\n            \'zcett\': \'(CETT-fixed_REF_W_c_mu)/max(fixed_REF_W_c_sd,1e-6)\'}\n\n\ndef verified_request(output, request_id, plan_hash):\n    directory = output / \'requests\' / request_id\n    if not directory.exists():\n        return None\n    record = json.loads((directory / \'result.json\').read_text())\n    if (record.get(\'state\') != \'complete\' or record.get(\'plan_sha256\') != plan_hash\n            or record.get(\'request_id\') != request_id or len(record.get(\'arms\', [])) != 5\n            or [row[\'arm\'] for row in record[\'arms\']] != list(ARMS)):\n        raise ValueError(\'Existing request does not match fixed all-five-arm plan: \' + request_id)\n    proof = record.get(\'technical_checks\')\n    if not isinstance(proof, dict) or proof.get(\'state\') != \'passed\' or proof.get(\'plan_sha256\') != plan_hash:\n        raise ValueError(\'Request lacks bound technical-check evidence: \' + request_id)\n    if sha_file(directory / \'arrays.npz\') != record[\'npz_sha256\']:\n        raise ValueError(\'Existing request array SHA mismatch: \' + request_id)\n    return record\n\n\ndef restore_resume(resume_zip, output, plan_hash):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        source_plan = json.loads(bundle.read(\'run_plan.json\'))\n        if hashlib.sha256(canonical(source_plan)).hexdigest() != plan_hash:\n            raise ValueError(\'Resume ZIP belongs to a different immutable plan/reference\')\n        for index in range(16):\n            request_id = f\'r{index:03d}\'\n            prefix = f\'requests/{request_id}/\'\n            names = [prefix + \'result.json\', prefix + \'arrays.npz\']\n            present = [name in bundle.namelist() for name in names]\n            if not any(present):\n                continue\n            if not all(present):\n                raise ValueError(\'Resume ZIP contains an incomplete request chunk\')\n            destination = output / \'requests\' / request_id\n            if destination.exists():\n                verified_request(output, request_id, plan_hash)\n                if any((destination / Path(name).name).read_bytes() != bundle.read(name) for name in names):\n                    raise ValueError(\'Resume ZIP conflicts with existing complete request; no overwrite\')\n                continue\n            parent = destination.parent\n            parent.mkdir(parents=True, exist_ok=True)\n            temporary = Path(tempfile.mkdtemp(prefix=\'.\' + request_id + \'_\', dir=parent))\n            try:\n                for name in names:\n                    if bundle.getinfo(name).file_size > 128 * (1 << 20):\n                        raise ValueError(\'Oversized request chunk in resume ZIP\')\n                    (temporary / Path(name).name).write_bytes(bundle.read(name))\n                record = json.loads((temporary / \'result.json\').read_text())\n                if record.get(\'plan_sha256\') != plan_hash or sha_file(temporary / \'arrays.npz\') != record.get(\'npz_sha256\'):\n                    raise ValueError(\'Resume request failed hash/plan verification\')\n                os.rename(temporary, destination)\n                verified_request(output, request_id, plan_hash)\n            finally:\n                if temporary.exists():\n                    shutil.rmtree(temporary)\n\n\ndef write_request(output, request, results, arrays, plan_hash, hardware, technical_checks=None):\n    import numpy as np\n    destination = output / \'requests\' / request[\'request_id\']\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        raise ValueError(\'Refusing to overwrite existing request outputs\')\n    temporary = Path(tempfile.mkdtemp(dir=destination.parent, prefix=\'.\' + destination.name + \'_\'))\n    try:\n        np.savez_compressed(temporary / \'arrays.npz\',\n                            **{key: np.stack([row[key] for row in arrays]) for key in arrays[0]})\n        with (temporary / \'arrays.npz\').open(\'rb\') as stream:\n            os.fsync(stream.fileno())\n        public_request = {key: value for key, value in request.items() if key != \'ids\'}\n        record = {\'state\': \'complete\', \'request_id\': request[\'request_id\'], \'request\': public_request,\n                  \'plan_sha256\': plan_hash, \'npz_sha256\': sha_file(temporary / \'arrays.npz\'), \'arms\': results,\n                  \'hardware\': hardware, \'technical_checks\': technical_checks, \'completed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        atomic_json(temporary / \'result.json\', record)\n        os.rename(temporary, destination)  # Both files become visible as one paired-request checkpoint.\n        return record\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n\n\ndef run(hn_set, policy, refw, items_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'completed_requests\': 0,\n              \'confirmatory_eligible\': False, \'scope\': \'original_fixed_H286_exploratory_pilot\'}\n    metrics = {\'pretrained_model_run\': False, \'checks\': {}, \'no_training\': True}\n    started = time.monotonic()\n    def save():\n        atomic_json(output / \'status.json\', status)\n        atomic_json(output / \'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items = load_inputs(hn_set, policy, refw, items_path)\n        plan = plan_document(reference, stats, items)\n        plan_hash = hashlib.sha256(canonical(plan)).hexdigest()\n        existing = output / \'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != canonical(plan) + b\'\\n\':\n            raise ValueError(\'Existing output uses a different plan; choose a fresh directory\')\n        if not existing.exists():\n            atomic_json(existing, plan)\n        restore_resume(resume_zip, output, plan_hash)\n        done = [verified_request(output, f\'r{index:03d}\', plan_hash) for index in range(len(items))]\n        status[\'completed_requests\'] = sum(item is not None for item in done)\n        if all(item is not None for item in done):\n            metrics[\'technical_checks_from_completed_requests\'] = [row.get(\'technical_checks\') for row in done]\n            metrics[\'checks_executed_this_attempt\'] = False\n            if any(not isinstance(row.get(\'technical_checks\'), dict) or row[\'technical_checks\'].get(\'state\') != \'passed\'\n                   or row[\'technical_checks\'].get(\'plan_sha256\') != plan_hash for row in done):\n                raise ValueError(\'Completed resume lacks the bound technical-check evidence\')\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True)\n            return status\n        from importlib.metadata import version\n\n        import torch\n        import transformers\n        if transformers.__version__ != TRANSFORMERS or version(\'accelerate\') != ACCELERATE:\n            raise ValueError(\'Exact transformers5.22.2 and accelerate1.16.0 are required\')\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.set_float32_matmul_precision(\'highest\')\n        torch.manual_seed(SEED)\n        disable_bar = getattr(transformers.utils.logging, \'disable_progress_bar\', None)\n        if disable_bar is not None:\n            disable_bar()\n        metrics.update(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version())\n        status[\'stage\'] = \'hardware_preflight\'\n        cache = Path(cache_dir or Path.home() / \'.cache\' / \'h286_pilot\')\n        hardware = hardware_preflight(torch, cache)\n        metrics[\'hardware\'] = hardware\n        status[\'stage\'] = \'pinned_download_and_load\'\n        save()\n        os.environ[\'HF_DEACTIVATE_ASYNC_LOAD\'] = \'1\'\n        print(\'Status: Original Qwen3-1.7B laden, exakte Dateien prüfen; FP32/SDPA.\', flush=True)\n        tokenizer, model = load_model(reference, cache)\n        metrics[\'weights_loaded\'] = True\n        hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(), compute_capability=list(torch.cuda.get_device_capability(0)), transformers=transformers.__version__, attention=\'sdpa/EFFICIENT_ATTENTION\', dtype=\'float32\', tf32=False)\n        groups = defaultdict(list)\n        for layer, neuron in reference.coordinates:\n            groups[layer].append(neuron)\n        with torch.inference_mode():\n            weight_norms = {layer: model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[columns]\n                            for layer, columns in groups.items()}\n        evidence = parameter_evidence(model)\n        first_index = next(index for index, result in enumerate(done) if result is None)\n        warm = prepare_request(tokenizer, first_index, items[first_index])\n        ids = warm[\'ids\'].to(\'cuda:0\')\n        status[\'stage\'] = \'warmup\'\n        save()\n        with torch.inference_mode(), torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION):\n            model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)  # untimed warmup\n            baseline = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.detach().cpu()\n        metrics[\'pretrained_model_run\'] = True\n        status[\'stage\'] = \'technical_checks\'\n        print(\'Status: Zwei technische Prüfungen vor der festen 16-Request-Auswertung.\', flush=True)\n        _sham, sham_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[0], include_logits=True, weight_norms=weight_norms)\n        selected_baseline = baseline[:, warm[\'answer_logit_positions\']].float()\n        observed = torch.from_numpy(sham_arrays[\'technical_logits\'])\n        delta = (selected_baseline - observed).abs()\n        metrics[\'checks\'][\'sham\'] = {\'bitwise_equal\': torch.equal(selected_baseline, observed),\n                                     \'max_abs_logit_difference\': float(delta.max()), \'rtol\': 1e-5, \'atol\': 1e-6}\n        if not torch.allclose(selected_baseline, observed, rtol=1e-5, atol=1e-6):\n            raise ValueError(\'Technical sham logit check failed; exploratory queue not started\')\n        pulse_result, pulse_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[1], weight_norms=weight_norms)\n        import numpy as np\n        upstream = [index for index, (layer, _) in enumerate(reference.coordinates) if layer < 14]\n        earlier = [index for index, position in enumerate(warm[\'capture_positions\']) if position < warm[\'pulse_positions\'][0]]\n        upstream_equal = np.array_equal(sham_arrays[\'raw_z\'][..., upstream], pulse_arrays[\'raw_z\'][..., upstream])\n        earlier_equal = np.array_equal(sham_arrays[\'raw_z\'][:, earlier], pulse_arrays[\'raw_z\'][:, earlier])\n        metrics[\'checks\'][\'causality\'] = {\'upstream_coordinates\': len(upstream), \'upstream_raw_equal\': upstream_equal,\n                                           \'earlier_positions\': len(earlier), \'earlier_raw_equal\': earlier_equal}\n        if not upstream_equal or not earlier_equal:\n            raise ValueError(\'Technical causal guard failed; exploratory queue not started\')\n        metrics[\'checks\'][\'pulse_and_capture\'] = {\'pulse_once\': pulse_result[\'pulse_fired\'] == 1,\n                                                  \'capture_once\': pulse_result[\'capture_fired\'] == 1,\n                                                  \'frozen_parameter_evidence\': parameter_evidence(model) == evidence}\n        if parameter_evidence(model) != evidence:\n            raise ValueError(\'Model parameter identity/storage/version counters changed\')\n        metrics[\'checks_executed_this_attempt\'] = True\n        technical_checks = {\'state\': \'passed\', \'plan_sha256\': plan_hash, \'checks\': metrics[\'checks\'],\n                            \'hardware\': hardware, \'executed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        measured = []\n        for index, item in enumerate(items):\n            if done[index] is not None:\n                continue\n            request = prepare_request(tokenizer, index, item)\n            status.update(stage=\'exploratory_requests\', current_request=request[\'request_id\'])\n            save()\n            print(f"Request {index + 1}/16: {request[\'request_id\']} — fünf feste Arme", flush=True)\n            t0 = time.monotonic()\n            order = list(range(len(ARMS)))\n            random.Random(SEED + index).shuffle(order)\n            request[\'execution_order\'] = [ARMS[arm_index][\'name\'] for arm_index in order]\n            results, array_rows = [None] * len(ARMS), [None] * len(ARMS)\n            cpu_rng, cuda_rng = torch.random.get_rng_state().clone(), torch.cuda.get_rng_state().clone()\n            for arm_index in order:\n                result, arrays = run_arm(model, request, reference.coordinates, stats, ARMS[arm_index], weight_norms=weight_norms)\n                results[arm_index], array_rows[arm_index] = result, arrays\n            if not torch.equal(cpu_rng, torch.random.get_rng_state()) or not torch.equal(cuda_rng, torch.cuda.get_rng_state()):\n                raise ValueError(\'Global RNG changed during fixed five-arm request\')\n            if parameter_evidence(model) != evidence:\n                raise ValueError(\'Frozen parameter evidence changed; request is not published complete\')\n            record = write_request(output, request, results, array_rows, plan_hash, hardware, technical_checks)\n            done[index] = record\n            measured.append(time.monotonic() - t0)\n            status[\'completed_requests\'] = sum(row is not None for row in done)\n            remaining = len(items) - status[\'completed_requests\']\n            eta_seconds = sum(measured) / len(measured) * remaining\n            finish = dt.datetime.now(ZoneInfo(\'Europe/Berlin\')) + dt.timedelta(seconds=eta_seconds)\n            status.update(estimated_remaining_seconds=eta_seconds, estimated_finish_berlin=finish.isoformat())\n            save()\n            print(f"Gespeichert {status[\'completed_requests\']}/16; Restschätzung {eta_seconds / 60:.1f} min; Berlin {finish:%H:%M}", flush=True)\n            if remaining and (eta_seconds > 3600 or finish.date() > dt.datetime.now(ZoneInfo(\'Europe/Berlin\')).date()):\n                status.update(state=\'paused\', stage=\'safe_request_checkpoint\',\n                              reason=\'Runtime estimate exceeds one hour or today; resume ZIP keeps paired requests together\')\n                break\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'weights_unchanged_evidence\'] = \'same parameter identities, storage pointers, version counters; no training; no full tensor rehash\'\n        metrics[\'hardware_comparison_warning\'] = \'Each complete paired request stays on one GPU; resumed chunks can differ by GPU/software.\'\n    except BaseException as exc:  # noqa: BLE001 — preserve diagnostics also on interruption\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output / \'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \' + str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic() - started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--hn-set\', required=True)\n    parser.add_argument(\'--policy\', required=True)\n    parser.add_argument(\'--refw\', required=True)\n    parser.add_argument(\'--items\', required=True)\n    parser.add_argument(\'--out-dir\', required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    args = parser.parse_args()\n    result = run(args.hn_set, args.policy, args.refw, args.items, args.out_dir, args.cache_dir, args.resume_zip)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\'] in (\'complete\', \'paused\') else 1)\n', 'qwen3_h286_validation_runtime.py': '"""Fixed H286 answer-format validation: original teacher forcing plus greedy KV decode.\n\nThis is a separate plan/output format. Original reference files, FP32 numerics,\nteacher-forced measurements and five arms are reused without changing the old\nruntime. The 15-minute budget covers the experimental queue, checked only after\ncomplete all-five-arm request checkpoints; model loading/checks are additional.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\n\nimport qwen3_h286_runtime as base\n\nSCHEMA = "qwen3_h286_answer_validation_v1"\nPROOF_SCHEMA = "qwen3_h286_answer_validation_checks_v1"\nMAX_NEW_TOKENS, TOP_K, QUEUE_BUDGET_SECONDS = 48, 10, 900\nPROOF_CHECKS = ("teacher_forced_sham", "teacher_forced_causality", "generation_uninstrumented_prefill",\n                "generation_no_pulse_vs_sham", "pulse_capture_counts", "parameters_and_hook_cleanup")\n\n\ndef generation_module():\n    from attention_probe import answer_validation\n    return answer_validation\n\n\ndef plan_document(reference, stats, items):\n    previous = base.plan_document(reference, stats, items)\n    plan = dict(previous)\n    plan.update(schema=SCHEMA, purpose="original_fixed_H286_teacher_forcing_and_answer_format_validation",\n        previous_plan_sha256=hashlib.sha256(base.canonical(previous)).hexdigest(),\n        base_runtime_sha256=base.sha_file(base.__file__), helper_sha256=base.sha_file(__file__),\n        generation_module_sha256=base.sha_file(generation_module().__file__),\n        generation={"max_new_tokens": MAX_NEW_TOKENS, "top_k": TOP_K, "decoding": "greedy argmax",\n            "input": "prompt_ids only; never the teacher-forced target input",\n            "cache": "fresh cache per arm; one prefill pulse; hooks removed before cached decode",\n            "eos": "selected EOS counts toward cap; EOS is logged but not emitted",\n            "classification": "exact / format_equivalent / noncanonical / truncated; not semantic accuracy"},\n        array_contract={"tf": "unchanged original teacher-forced arrays","gen": "prompt-prefill activation arrays only",\n                            "variable_generation": "JSON only; no object/ragged NPZ arrays"},\n        queue_budget={"seconds": QUEUE_BUDGET_SECONDS, "scope": "experimental paired-request queue only",\n            "checkpoint": "complete all-five-arm request", "pause": "elapsed >= budget or remaining ETA > budget",\n            "limitation": "boundary checked; a slow complete request can exceed the budget; setup is additional"},\n        technical_proof_schema=PROOF_SCHEMA)\n    plan["module_sha256"] = dict(previous["module_sha256"],\n        **{"answer_validation.py": plan["generation_module_sha256"]})\n    return plan\n\n\ndef attention_context(torch, efficient_gpu):\n    backend = torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION if efficient_gpu else torch.nn.attention.SDPBackend.MATH\n    return torch.nn.attention.sdpa_kernel(backend)\n\n\ndef hook_count(model):\n    return sum(len(module._forward_hooks) + len(module._forward_pre_hooks) for module in model.modules())\n\n\ndef verify_proof(proof, plan_hash):\n    if (not isinstance(proof, dict) or proof.get("schema") != PROOF_SCHEMA\n            or proof.get("state") != "passed" or proof.get("plan_sha256") != plan_hash\n            or any(proof.get("checks", {}).get(name, {}).get("passed") is not True for name in PROOF_CHECKS)):\n        raise ValueError("Missing or incomplete bound answer-validation technical evidence")\n\n\ndef technical_checks(model, tokenizer, request, reference, stats, plan_hash, hardware, *,\n                     weight_norms=None, efficient_gpu=True):\n    """Native TF/prefill controls; full generation comparison is capture-instrumented no-pulse vs sham."""\n    import numpy as np\n    import torch\n    generation = generation_module().run_generation_arm\n    evidence, hooks = base.parameter_evidence(model), hook_count(model)\n    checks = {}\n    device = next(model.parameters()).device\n    with torch.inference_mode(), attention_context(torch, efficient_gpu):\n        full_ids = request["ids"].to(device)\n        native = model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids), use_cache=False).logits\n        native_tf = native[:, request["answer_logit_positions"]].detach().float().cpu()\n        del native\n        tf_sham, a_sham = base.run_arm(model, request, reference.coordinates, stats, base.ARMS[0],\n            efficient_gpu=efficient_gpu, include_logits=True, weight_norms=weight_norms)\n        observed = torch.from_numpy(a_sham["technical_logits"])\n        checks["teacher_forced_sham"] = {"passed": bool(torch.equal(native_tf, observed)),\n            "bitwise_equal": bool(torch.equal(native_tf, observed)), "max_abs_logit_difference": float((native_tf-observed).abs().max())}\n        tf_pulse, a_pulse = base.run_arm(model, request, reference.coordinates, stats, base.ARMS[1],\n            efficient_gpu=efficient_gpu, weight_norms=weight_norms)\n        upstream = [i for i, (layer, _) in enumerate(reference.coordinates) if layer < 14]\n        earlier = [i for i, position in enumerate(request["capture_positions"]) if position < request["pulse_positions"][0]]\n        up_equal = np.array_equal(a_sham["raw_z"][..., upstream], a_pulse["raw_z"][..., upstream])\n        early_equal = np.array_equal(a_sham["raw_z"][:, earlier], a_pulse["raw_z"][:, earlier])\n        checks["teacher_forced_causality"] = {"passed": bool(up_equal and early_equal),\n            "upstream_coordinates": len(upstream), "upstream_raw_equal": up_equal, "earlier_positions": len(earlier), "earlier_raw_equal": early_equal}\n        prompt = torch.tensor([request["prompt_ids"]], dtype=torch.long, device=device)\n        plain_output = model(input_ids=prompt, attention_mask=torch.ones_like(prompt), use_cache=True)\n        plain_logits = plain_output.logits[0, -1].detach().float().cpu().contiguous()\n        plain_hash = hashlib.sha256(plain_logits.numpy().tobytes()).hexdigest()\n        del plain_output, plain_logits\n        no_pulse, no_pulse_arrays = generation(model, tokenizer, request, reference.coordinates, stats, None,\n            efficient_gpu=efficient_gpu, weight_norms=weight_norms, max_new_tokens=MAX_NEW_TOKENS, top_k=TOP_K)\n        sham, sham_arrays = generation(model, tokenizer, request, reference.coordinates, stats, base.ARMS[0],\n            efficient_gpu=efficient_gpu, weight_norms=weight_norms, max_new_tokens=MAX_NEW_TOKENS, top_k=TOP_K)\n    prefix_equal = plain_hash == no_pulse["first_step_logits_sha256"] == sham["first_step_logits_sha256"]\n    checks["generation_uninstrumented_prefill"] = {"passed": prefix_equal,\n        "plain_vs_captured_first_logits_bitwise_equal": prefix_equal, "first_step_logits_sha256": plain_hash}\n    validate_generation(sham, request, base.ARMS[0])\n    sequence_keys = ("generated_token_ids", "selected_token_ids", "generated_text", "steps", "stop_reason", "truncated", "first_token_gold",\n                     "prompt_length", "prefill_forward_count", "decode_forward_count", "final_cache_length")\n    seq_equal = all(no_pulse[key] == sham[key] for key in sequence_keys)\n    arrays_equal = set(no_pulse_arrays) == set(sham_arrays) and all(\n        np.array_equal(no_pulse_arrays[key], sham_arrays[key]) for key in no_pulse_arrays)\n    checks["generation_no_pulse_vs_sham"] = {"passed": bool(seq_equal and arrays_equal),\n        "tokens_and_step_logprobs_equal": seq_equal, "prefill_arrays_equal": arrays_equal,\n        "control_scope": "Both full-decode controls have passive prefill capture; only the separate native prefill is fully unhooked."}\n    counts_ok = (tf_sham["pulse_fired"] == tf_sham["capture_fired"] == tf_pulse["pulse_fired"] == tf_pulse["capture_fired"] == 1\n        and no_pulse["pulse_fired"] == 0 and sham["pulse_fired"] == 1\n        and no_pulse["capture_fired"] == sham["capture_fired"] == 1\n        and no_pulse["prefill_forward_count"] == sham["prefill_forward_count"] == 1)\n    finite = all(np.isfinite(value).all() for recorded in (a_sham, a_pulse, no_pulse_arrays, sham_arrays)\n                 for value in recorded.values())\n    checks["pulse_capture_counts"] = {"passed": bool(counts_ok and finite), "all_recordings_finite": bool(finite)}\n    frozen, clean = base.parameter_evidence(model) == evidence, hook_count(model) == hooks\n    checks["parameters_and_hook_cleanup"] = {"passed": bool(frozen and clean),\n        "frozen_parameter_evidence": frozen, "hook_count_before": hooks, "hook_count_after": hook_count(model)}\n    proof = {"schema": PROOF_SCHEMA, "state": "passed", "plan_sha256": plan_hash, "checks": checks,\n                 "hardware": hardware, "executed_utc": base.dt.datetime.now(base.dt.UTC).isoformat()}\n    verify_proof(proof, plan_hash)\n    return proof\n\n\ndef validate_generation(record, request, arm):\n    if (record.get("schema") != "h286_prompt_only_generation_v1" or record.get("arm") != arm\n        or record.get("request_id") != request["request_id"] or record.get("max_new_tokens") != MAX_NEW_TOKENS\n        or record.get("top_k") != TOP_K or record.get("prompt_length") != len(request["prompt_ids"])\n        or record.get("prefill_forward_count") != 1 or record.get("pulse_fired") != 1 or record.get("capture_fired") != 1):\n        raise ValueError("Generation record does not match the fixed prefill/arm protocol")\n    positions = list(range(max(0, len(request["prompt_ids"])-4), len(request["prompt_ids"])))\n    if record.get("prefill_capture_positions") != positions or record.get("pulse_positions") != [len(request["prompt_ids"])-1]:\n        raise ValueError("Generation capture must use prompt positions only")\n    selected, generated, steps = (record.get(key) for key in ("selected_token_ids", "generated_token_ids", "steps"))\n    if (not isinstance(selected, list) or not 1 <= len(selected) <= MAX_NEW_TOKENS\n        or not isinstance(generated, list) or not isinstance(steps, list) or len(steps) != len(selected)\n        or [step.get("selected_token_id") for step in steps] != selected\n        or record.get("decode_forward_count") != len(selected)-1\n        or record.get("final_cache_length") != len(request["prompt_ids"])+len(selected)-1):\n        raise ValueError("Generation token/cache counts are inconsistent")\n    eos = record.get("stop_reason") == "eos"\n    capped = record.get("stop_reason") == "max_new_tokens"\n    if (not (eos or capped) or record.get("truncated") is not capped\n        or generated != (selected[:-1] if eos else selected)\n        or (capped and len(selected) != MAX_NEW_TOKENS)\n        or (eos and not steps[-1].get("is_eos"))\n        or (capped and any(step.get("is_eos") for step in steps))):\n        raise ValueError("EOS/cap/truncation convention is inconsistent")\n    if record.get("classification") not in ("exact", "format_equivalent", "noncanonical", "truncated"):\n        raise ValueError("Unknown surface classification")\n    if capped and record["classification"] != "truncated":\n        raise ValueError("A capped output must be classified as truncated")\n    for step in steps:\n        if (not math.isfinite(step["selected_token_logprob"]) or not isinstance(step.get("top_tokens"), list)\n                or not 1 <= len(step["top_tokens"]) <= TOP_K\n                or any(not math.isfinite(token["logprob"]) for token in step["top_tokens"])):\n            raise ValueError("Missing or nonfinite generated-token/top-k log probabilities")\n\n\ndef validate_payload(request, results, arrays, coordinate_count):\n    import numpy as np\n    if len(results) != len(base.ARMS) or len(arrays) != len(base.ARMS):\n        raise ValueError("A checkpoint needs all five arms")\n    expected_keys = None\n    expected_shapes = None\n    for arm, result, values in zip(base.ARMS, results, arrays):\n        if result.get("arm") != arm or not isinstance(result.get("teacher_forced"), dict) or not isinstance(result.get("generation"), dict):\n            raise ValueError("Both teacher-forced and generation branches are required for every arm")\n        tf = result["teacher_forced"]\n        if tf.get("arm") != arm or tf.get("pulse_fired") != 1 or tf.get("capture_fired") != 1:\n            raise ValueError("Teacher-forced pulse/capture evidence is incomplete")\n        probabilities = tf.get("answer_token_logprobs", [])\n        if len(probabilities) != len(request["answer_ids"]) or not probabilities or any(not math.isfinite(p) for p in probabilities):\n            raise ValueError("Teacher-forced answer probabilities are missing/nonfinite")\n        validate_generation(result["generation"], request, arm)\n        required = {"tf_raw_z", "tf_cett", "tf_zcett", "tf_down_output_norm_raw", "tf_gold_token_logprob",\n                    "gen_raw_z", "gen_cett", "gen_zcett", "gen_down_output_norm_raw"}\n        required.update(f"{prefix}residual_layer_{layer}" for prefix in ("tf_", "gen_") for layer in (3, 7, 14, 27))\n        if not required.issubset(values) or any(not key.startswith(("tf_", "gen_")) for key in values):\n            raise ValueError("Missing or unprefixed validation activation arrays")\n        shapes = {key: tuple(np.asarray(value).shape) for key, value in values.items()}\n        if expected_keys is not None and (set(values) != expected_keys or shapes != expected_shapes):\n            raise ValueError("All five arms need identical fixed NPZ keys/shapes")\n        expected_keys, expected_shapes = set(values), shapes\n        for value in values.values():\n            a = np.asarray(value)\n            if a.dtype != np.float32 or not np.isfinite(a).all():\n                raise ValueError("All published activation arrays must be finite FP32")\n        for prefix, positions in (("tf_", request["capture_positions"]), ("gen_", result["generation"]["prefill_capture_positions"])):\n            expected = (1, len(positions), coordinate_count)\n            if any(shapes[prefix+key] != expected for key in ("raw_z", "cett", "zcett", "down_output_norm_raw")):\n                raise ValueError("Activation shapes do not match the ordered fixed H286 coordinates")\n            width = shapes["tf_residual_layer_3"][-1]\n            if width < 1 or any(shapes[f"{prefix}residual_layer_{layer}"] != (1, len(positions), width) for layer in (3, 7, 14, 27)):\n                raise ValueError("Residual arrays must share hidden width and the correct branch positions")\n        if (shapes["tf_gold_token_logprob"] != (1, len(request["answer_ids"]))\n            or not np.array_equal(values["tf_gold_token_logprob"], np.asarray(probabilities, dtype=np.float32)[None, :])):\n            raise ValueError("Teacher-forced saved token probabilities disagree with JSON")\n\n\ndef compute_request(model, tokenizer, request, reference, stats, *, weight_norms=None, efficient_gpu=True):\n    import torch\n    runner = generation_module().run_generation_arm\n    results, arrays = [None]*5, [None]*5\n    order = list(range(5)); random.Random(base.SEED + int(request["request_id"][1:])).shuffle(order)\n    request["execution_order"] = [base.ARMS[i]["name"] for i in order]\n    request["validation_schema"] = SCHEMA\n    for i in order:\n        arm = base.ARMS[i]\n        with attention_context(torch, efficient_gpu):\n            tf, tf_arrays = base.run_arm(model, request, reference.coordinates, stats, arm,\n                                         efficient_gpu=efficient_gpu, weight_norms=weight_norms)\n            gen, gen_arrays = runner(model, tokenizer, request, reference.coordinates, stats, arm,\n                efficient_gpu=efficient_gpu, weight_norms=weight_norms, max_new_tokens=MAX_NEW_TOKENS, top_k=TOP_K)\n        results[i] = {"arm": arm, "teacher_forced": tf, "generation": gen}\n        arrays[i] = {**{"tf_"+key: value for key, value in tf_arrays.items()},\n                     **{"gen_"+key: value for key, value in gen_arrays.items()}}\n    validate_payload(request, results, arrays, len(reference.coordinates))\n    return results, arrays\n\n\ndef verified_request(output, request_id, plan_hash, coordinate_count):\n    import numpy as np\n    record = base.verified_request(output, request_id, plan_hash)\n    if record is None:\n        return None\n    request = record["request"]\n    if request.get("validation_schema") != SCHEMA:\n        raise ValueError("Old teacher-forcing chunks cannot be resumed as answer validation")\n    verify_proof(record["technical_checks"], plan_hash)\n    with np.load(Path(output)/"requests"/request_id/"arrays.npz", allow_pickle=False) as packed:\n        if any(packed[key].shape[0] != 5 for key in packed.files):\n            raise ValueError("Saved NPZ lacks the five-arm dimension")\n        arrays = [{key: packed[key][i] for key in packed.files} for i in range(5)]\n    validate_payload(request, record["arms"], arrays, coordinate_count)\n    return record\n\n\ndef restore_resume(path, output, plan_hash, coordinate_count):\n    if not path:\n        return\n    with zipfile.ZipFile(path) as packed:\n        document = json.loads(packed.read("run_plan.json"))\n        if document.get("schema") != SCHEMA or hashlib.sha256(base.canonical(document)).hexdigest() != plan_hash:\n            raise ValueError("Resume ZIP has another validation plan/schema")\n    # The old importer copies paired chunks atomically and verifies the bound NPZ\n    # digest. Strengthen every imported row before it can be counted as complete.\n    base.restore_resume(path, output, plan_hash)\n    for i in range(16):\n        verified_request(output, f"r{i:03d}", plan_hash, coordinate_count)\n\n\ndef archive(output):\n    output = Path(output); target = output/"qwen3_h286_validation_results.zip"\n    temporary = output/".qwen3_h286_validation_results.zip.tmp"\n    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as packed:\n        for path in sorted(output.rglob("*")):\n            if path.is_file() and path not in (target, temporary) and path.suffix != ".zip" and not any(\n                    part.startswith(".") for part in path.relative_to(output).parts):\n                packed.write(path, path.relative_to(output))\n    os.replace(temporary, target)\n    return target\n\n\ndef initialize_engine(reference, cache, *, on_loaded=None):\n    from importlib.metadata import version\n\n    import torch\n    import transformers\n    if transformers.__version__ != base.TRANSFORMERS or version("accelerate") != base.ACCELERATE:\n        raise ValueError("Exact transformers5.22.2 and accelerate1.16.0 are required")\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.set_float32_matmul_precision("highest"); torch.manual_seed(base.SEED)\n    transformers.utils.logging.disable_progress_bar()\n    hardware = base.hardware_preflight(torch, cache)\n    os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"\n    print("Status: Original FP32 Qwen3-1.7B laden; feste Referenz und neue Antwortvalidierung.", flush=True)\n    tokenizer, model = base.load_model(reference, cache)\n    hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(),\n        compute_capability=list(torch.cuda.get_device_capability(0)), transformers=transformers.__version__,\n        dtype="float32", tf32=False, attention="sdpa/EFFICIENT_ATTENTION")\n    if on_loaded is not None:\n        on_loaded(hardware)\n    groups = {}\n    for layer, channel in reference.coordinates:\n        groups.setdefault(layer, []).append(channel)\n    with torch.inference_mode():\n        norms = {layer:model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[channels]\n                 for layer, channels in groups.items()}\n    return {"model": model, "tokenizer": tokenizer, "hardware": hardware, "weight_norms": norms}\n\n\ndef run_queue(engine, reference, stats, items, output, plan_hash, proof, done, save, *, clock=None, efficient_gpu=True):\n    import torch\n    clock = clock or time.monotonic\n    started = clock(); measured = []\n    model, tokenizer = engine["model"], engine["tokenizer"]\n    hardware = engine["hardware"]; evidence = base.parameter_evidence(model)\n    verify_proof(proof, plan_hash)\n    status = {"state": "running", "stage": "answer_validation_requests", "completed_requests": sum(r is not None for r in done)}\n    for i, item in enumerate(items):\n        if done[i] is not None:\n            continue\n        request = base.prepare_request(tokenizer, i, item)\n        status.update(current_request=request["request_id"]); save(status)\n        print(f"Request {i+1}/16: {request[\'request_id\']} — fünf Arme, je Teacher-Forcing plus freie Greedy-Antwort", flush=True)\n        request_started = clock()\n        cpu_rng = torch.random.get_rng_state().clone()\n        cuda_rng = torch.cuda.get_rng_state().clone() if efficient_gpu else None\n        result, arrays = compute_request(model, tokenizer, request, reference, stats,\n            weight_norms=engine.get("weight_norms"), efficient_gpu=efficient_gpu)\n        validate_payload(request, result, arrays, len(reference.coordinates))\n        if not torch.equal(cpu_rng, torch.random.get_rng_state()) or (cuda_rng is not None and not torch.equal(cuda_rng, torch.cuda.get_rng_state())):\n            raise ValueError("Global RNG changed during a paired request")\n        if base.parameter_evidence(model) != evidence:\n            raise ValueError("Frozen parameter evidence changed; request not published")\n        record = base.write_request(output, request, result, arrays, plan_hash, hardware, proof)\n        done[i] = record\n        measured.append(clock()-request_started)\n        remaining = sum(row is None for row in done)\n        eta = sum(measured)/len(measured)*remaining\n        elapsed = clock()-started\n        status.update(completed_requests=len(done)-remaining, experimental_queue_seconds=elapsed,\n                      estimated_remaining_seconds=eta)\n        save(status)\n        print(f"Gespeichert {status[\'completed_requests\']}/16; Restschätzung {eta/60:.1f} min; Queue {elapsed/60:.1f} min", flush=True)\n        if remaining and (elapsed >= QUEUE_BUDGET_SECONDS or eta > QUEUE_BUDGET_SECONDS):\n            status.update(state="paused",stage="safe_request_checkpoint",\n                reason="Experimental queue elapsed or remaining ETA reached 15-minute boundary; all five arms saved together")\n            save(status)\n            return status\n    status.update(state="complete", stage="complete"); save(status)\n    return status\n\n\ndef run(hn_set, policy, refw, items_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir); output.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    status = {"state": "running", "stage": "reference", "completed_requests": 0,\n                  "confirmatory_eligible": False, "scope": SCHEMA}\n    metrics = {"weights_loaded": False, "weights_loaded_this_attempt": False, "pretrained_model_run": False,\n                   "wrapper_attempt_id": os.environ.get("H286_WRAPPER_ATTEMPT"),\n                   "checks_executed_this_attempt": False, "checks": {}, "no_training": True,\n                   "purpose": "teacher-forcing comparison and prompt-only answer-format validation"}\n    def save(extra=None):\n        if extra:status.update(extra)\n        base.atomic_json(output/"status.json",status); base.atomic_json(output/"metrics.json",metrics)\n    save()\n    try:\n        reference, stats, items = base.load_inputs(hn_set, policy, refw, items_path)\n        plan = plan_document(reference, stats, items); plan_hash = hashlib.sha256(base.canonical(plan)).hexdigest()\n        path = output/"run_plan.json"\n        if path.exists() and path.read_bytes() != base.canonical(plan)+b"\\n":\n            raise ValueError("Existing output has another plan; old pilot chunks cannot be mixed")\n        if not path.exists():base.atomic_json(path, plan)\n        restore_resume(resume_zip, output, plan_hash, len(reference.coordinates))\n        done = [verified_request(output,f"r{i:03d}",plan_hash,len(reference.coordinates)) for i in range(16)]\n        status["completed_requests"] = sum(r is not None for r in done)\n        if all(r is not None for r in done):\n            metrics.update(retained_request_proofs=[r["technical_checks"] for r in done],\n                           previous_complete_requests_verified=True)\n            status.update(state="complete",stage="complete",resumed_without_gpu=True)\n            return status\n        status.update(stage="hardware_preflight_and_pinned_model_load"); save()\n        def mark_loaded(hardware):\n            metrics.update(weights_loaded=True, weights_loaded_this_attempt=True, hardware=hardware)\n            save()\n        engine = initialize_engine(reference, Path(cache_dir or Path.home()/".cache/h286_pilot"), on_loaded=mark_loaded)\n        metrics.update(weights_loaded=True, weights_loaded_this_attempt=True, hardware=engine["hardware"])\n        request = base.prepare_request(engine["tokenizer"],next(i for i,r in enumerate(done) if r is None),\n                                       items[next(i for i,r in enumerate(done) if r is None)])\n        import torch\n        status.update(stage="warmup"); save()\n        ids = request["ids"].to(next(engine["model"].parameters()).device)\n        with torch.inference_mode(), attention_context(torch, True):\n            engine["model"](input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False)\n        metrics["pretrained_model_run"] = True\n        status.update(stage="technical_checks"); save()\n        print("Status: Teacher-Forcing-, Prompt-Prefill- und Greedy-Sham-Prüfungen.",flush=True)\n        proof = technical_checks(engine["model"],engine["tokenizer"],request,reference,stats,plan_hash,engine["hardware"],\n                                 weight_norms=engine["weight_norms"])\n        metrics.update(checks=proof["checks"],checks_executed_this_attempt=True)\n        status.update(run_queue(engine,reference,stats,items,output,plan_hash,proof,done,save))\n        metrics["weights_unchanged_evidence"] = "parameter identities, storage pointers, version counters; no training; not a full tensor rehash"\n        metrics["generation_scope"] = "prompt-only greedy continuation with fresh cache per arm; pulse/capture only on prefill"\n    except BaseException as error:  # noqa: BLE001 — interrupt paths must retain the diagnostic ZIP.\n        status.update(state="failed",error_type=type(error).__name__,error=str(error))\n        (output/"traceback.txt").write_text(traceback.format_exc())\n        print("HARD FAILURE: "+str(error),flush=True)\n    finally:\n        metrics["elapsed_seconds"] = time.monotonic()-started\n        save(); archive(output)\n    return status\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("hn-set","policy","refw","items","out-dir"):\n        parser.add_argument("--"+name,required=True)\n    parser.add_argument("--cache-dir");parser.add_argument("--resume-zip")\n    args = parser.parse_args()\n    result = run(args.hn_set,args.policy,args.refw,args.items,args.out_dir,args.cache_dir,args.resume_zip)\n    print(json.dumps(result),flush=True)\n    raise SystemExit(0 if result["state"] in ("complete","paused") else 1)\n', 'attention_probe/answer_validation.py': '"""Prompt-only greedy answer validation; the attention pulse ends after prefill.\n\nThe old teacher-forcing runner is unchanged. This module retains its fixed REF-W\nCETT definition but never supplies request[\'ids\'], input_ids or answer_ids to\ngeneration. All saved activation arrays contain prompt positions only.\n"""\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport re\nimport time\nfrom numbers import Integral\n\nimport torch\n\nfrom .interventions import AttentionPulse, CoordinateCapture, _cache_length\n\nSEED = 20261117\nMAX_NEW_TOKENS = 48\nTERMINAL_PUNCTUATION = ".!?,;:"\n\n\ndef normalize_answer(text: str) -> str:\n    """Only case, outer whitespace and one simple terminal punctuation mark."""\n    if not isinstance(text, str):\n        raise TypeError(\'Answer text must be a string\')\n    result = text.strip().casefold()\n    if result and result[-1] in TERMINAL_PUNCTUATION:\n        result = result[:-1].rstrip()\n    return result\n\n\ndef classify_answer(generated: str, expected: str, *, truncated: bool = False) -> dict:\n    """Strict surface comparison, never extract an answer from an explanation.\n\n    Numeric values are not parsed or rounded: \'15.0\', \'+15\' and \'015\' do not\n    match \'15\'. A capped output is always classified as truncated, even if its\n    current surface matches. Noncanonical is not a semantic-incorrect verdict.\n    """\n    normalized, gold = normalize_answer(generated), normalize_answer(expected)\n    if not gold:\n        raise ValueError(\'Expected answer must not be empty\')\n    surface = (\'exact\' if generated == expected else\n               \'format_equivalent\' if normalized == gold else \'noncanonical\')\n    return {"classification": \'truncated\' if truncated else surface,\n                "surface_classification": surface, "normalized_generated": normalized,\n                "normalized_expected": gold, "truncated": bool(truncated),\n                "normalization": \'casefold; outer whitespace; at most one final . ! ? , ; :\',\n                "interpretation": \'Surface comparison only; noncanonical is not a semantic-wrong label.\'}\n\n\ndef _decode(tokenizer, ids):\n    return tokenizer.decode(list(ids), skip_special_tokens=False, clean_up_tokenization_spaces=False)\n\n\ndef _eos_ids(model, vocab_size):\n    configured = getattr(getattr(model, \'generation_config\', None), \'eos_token_id\', None)\n    source = \'generation_config\'\n    if configured is None:\n        configured = getattr(model.config, \'eos_token_id\', None)\n        source = \'config\'\n    if configured is None:\n        return (), \'none\'\n    values = [configured] if isinstance(configured, Integral) else list(configured)\n    if any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in values):\n        raise ValueError(\'EOS IDs must be integer vocabulary IDs\')\n    return tuple(sorted({int(v) for v in values})), source\n\n\ndef _cett_context(model, coordinates, positions, weight_norms):\n    # The source bundle installs the old runner at its root. A repository checkout\n    # can instead resolve it as the notebooks namespace; neither path changes it.\n    try:\n        from qwen3_h286_runtime import cett_tap\n    except ModuleNotFoundError as error:\n        if error.name != \'qwen3_h286_runtime\':\n            raise\n        from notebooks.qwen3_h286_runtime import cett_tap\n    return cett_tap(model, coordinates, positions, weight_norms)\n\n\ndef _sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\n\ndef run_generation_arm(model, tokenizer, request, coordinates, stats, arm, *,\n                       efficient_gpu=True, weight_norms=None, max_new_tokens=48,\n                       top_k=10):\n    """Return a JSON-compatible record and fixed-shape prefill numpy arrays.\n\n    arm=None disables the pulse; passive prefill capture still runs. Only that\n    prefill has hooks. Decode uses its returned cache and newly selected tokens.\n    Selected EOS is logged in steps/selected_token_ids but excluded from emitted\n    generated_token_ids and generated_text. The 48-step maximum includes EOS.\n    """\n    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, Integral) or not 1 <= max_new_tokens <= MAX_NEW_TOKENS:\n        raise ValueError(\'max_new_tokens must be an integer from 1 through 48\')\n    if isinstance(top_k, bool) or not isinstance(top_k, Integral) or not 1 <= top_k <= 10:\n        raise ValueError(\'top_k must be an integer from 1 through 10\')\n    prompt = list(request[\'prompt_ids\'])  # Sole source of generation input IDs.\n    vocab_size = int(model.config.vocab_size)\n    if not prompt or any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in prompt):\n        raise ValueError(\'prompt_ids must be a nonempty list of integer vocabulary IDs\')\n    request_id = request[\'request_id\']\n    if not isinstance(request_id, str) or re.fullmatch(r\'r[0-9]+\', request_id) is None:\n        raise ValueError(\'request_id must have the original rNNN form\')\n    seed = SEED + int(request_id[1:])\n    coordinates = tuple(tuple(pair) for pair in coordinates)\n    device = next(model.parameters()).device\n    if any(parameter.is_floating_point() and (parameter.dtype != torch.float32 or parameter.device != device)\n           for parameter in model.parameters()):\n        raise ValueError(\'Generation validation requires all floating parameters in FP32 on one device\')\n    if efficient_gpu:\n        if device.type != \'cuda\':\n            raise ValueError(\'efficient_gpu=True requires a CUDA model\')\n        if getattr(model.config, \'_attn_implementation\', None) != \'sdpa\':\n            raise ValueError(\'The GPU validation requires the sdpa attention implementation\')\n        if torch.backends.cuda.matmul.allow_tf32:\n            raise ValueError(\'Disable TF32 before full-FP32 generation validation\')\n    model.eval()\n    prompt_length = len(prompt)\n    positions = list(range(max(0, prompt_length-4), prompt_length))\n    pulse_positions = [prompt_length-1]\n    mu = torch.as_tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n    sd = torch.as_tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32)\n    if mu.shape != (len(coordinates),) or sd.shape != mu.shape or not bool(torch.isfinite(mu).all() and torch.isfinite(sd).all()) or bool((sd < 0).any()):\n        raise ValueError(\'Fixed REF-W means/SDs must match the finite ordered coordinates\')\n    sd = sd.clamp_min(1e-6)\n    eos_ids, eos_source = _eos_ids(model, vocab_size)\n    ids = torch.tensor([prompt], dtype=torch.long, device=device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    pulse = None\n    steps, emitted, selected_ids = [], [], []\n    decode_forwards = 0\n    _sync(device)\n    started = time.monotonic()\n    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False), attention:\n        with contextlib.ExitStack() as hooks:\n            capture = hooks.enter_context(CoordinateCapture(model, coordinates, positions,\n                residual_layers=tuple(l for l in (3, 7, 14, 27) if l < len(model.model.layers)),\n                expected_count=len(coordinates)))\n            contributions = hooks.enter_context(_cett_context(model, coordinates, positions, weight_norms))\n            if arm is not None:\n                pulse = hooks.enter_context(AttentionPulse(model, arm[\'block\'], pulse_positions,\n                    arm[\'beta\'], arm[\'mode\'], seed=seed))\n            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Prefill did not return a cache\')\n            # Capture arrays are copied to CPU by the existing read-only hooks.\n            raw = capture.values\n            cett = contributions[\'cett\']\n            zcett = (cett-mu)/sd\n            if not all(bool(torch.isfinite(v).all()) for v in (raw, cett, zcett, contributions[\'denominator\'])):\n                raise ValueError(\'Nonfinite prefill raw/CETT/reference-normalized capture\')\n            arrays = {"raw_z": raw.numpy(), "cett": cett.numpy(), "zcett": zcett.numpy(),\n                          "down_output_norm_raw": contributions[\'denominator\'].numpy()}\n            arrays.update({f\'residual_layer_{l}\': value.numpy() for l, value in capture.residuals.items()})\n            logits = output.logits[0, -1].float()\n        # ExitStack has removed every pulse/capture hook before any decode call.\n        _sync(device)\n        prefill_seconds = time.monotonic()-started\n        if not bool(torch.isfinite(logits).all()):\n            raise ValueError(\'Nonfinite prefill logits\')\n        first_logits = logits.detach().cpu().clone()\n        first_logits_sha = hashlib.sha256(first_logits.numpy().tobytes()).hexdigest()\n        stop_reason = \'max_new_tokens\'\n        for step in range(int(max_new_tokens)):\n            if not bool(torch.isfinite(logits).all()):\n                raise ValueError(\'Nonfinite generation logits\')\n            logprobs = logits.log_softmax(-1)\n            chosen = int(torch.argmax(logits))\n            values, indices = torch.topk(logprobs, k=min(int(top_k), vocab_size))\n            top_tokens = [{"token_id": int(t), "text": _decode(tokenizer, [int(t)]), "logprob": float(v)}\n                          for t, v in zip(indices.detach().cpu().tolist(), values.detach().cpu().tolist())]\n            is_eos = chosen in eos_ids\n            steps.append({"step": step, "phase": \'prefill\' if step == 0 else \'decode\',\n                "logit_position": prompt_length-1+step, "selected_token_id": chosen,\n                "selected_token_text": _decode(tokenizer, [chosen]), "selected_token_logprob": float(logprobs[chosen]),\n                "is_eos": is_eos, "top_tokens": top_tokens})\n            selected_ids.append(chosen)\n            if is_eos:\n                stop_reason = \'eos\'\n                break\n            emitted.append(chosen)\n            if step+1 == max_new_tokens:\n                break\n            # Only the model\'s own last selected token is passed to cached decode.\n            current = torch.tensor([[chosen]], dtype=torch.long, device=device)\n            mask = torch.ones((1, prompt_length+len(emitted)), dtype=torch.long, device=device)\n            output = model(input_ids=current, attention_mask=mask, past_key_values=cache, use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Decode lost its cache\')\n            logits = output.logits[0, -1].float()\n            decode_forwards += 1\n        _sync(device)\n        elapsed = time.monotonic()-started\n    generated_text = _decode(tokenizer, emitted)\n    truncated = stop_reason == \'max_new_tokens\'\n    # Gold is consulted only after generation has ended, solely for diagnostics.\n    answer = request[\'answer\']\n    classification = classify_answer(generated_text, answer, truncated=truncated)\n    gold_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not gold_ids or not isinstance(gold_ids[0], Integral) or not 0 <= gold_ids[0] < vocab_size:\n        raise ValueError(\'Expected answer must tokenize to a valid first vocabulary ID\')\n    gold_first = int(gold_ids[0])\n    first_logprobs = first_logits.log_softmax(-1)\n    first_gold = {"token_id": gold_first, "text": _decode(tokenizer, [gold_first]),\n        "rank": int((first_logits > first_logits[gold_first]).sum())+1,\n        "rank_rule": \'one plus number of strictly higher logits; ties can share rank\',\n        "logprob": float(first_logprobs[gold_first]), "is_top1": int(torch.argmax(first_logits)) == gold_first}\n    record = {"schema": \'h286_prompt_only_generation_v1\', "request_id": request_id,\n        "arm": dict(arm) if arm is not None else None, "seed": seed, "prompt_length": prompt_length,\n        "generation_input_source": \'request.prompt_ids only; no teacher-forced answer input\',\n        "prefill_capture_positions": positions, "pulse_positions": pulse_positions if pulse is not None else [],\n        "max_new_tokens": int(max_new_tokens), "top_k": int(top_k), "eos_token_ids": list(eos_ids), "eos_source": eos_source,\n        "generated_token_ids": emitted, "generated_text": generated_text, "selected_token_ids": selected_ids,\n        "eos_convention": \'selected EOS is recorded in steps/selected_token_ids but not emitted text/IDs\',\n        "steps": steps, "num_steps": len(steps), "num_generated_tokens": len(emitted), "stop_reason": stop_reason,\n        "truncated": truncated, "classification": classification[\'classification\'], "classification_details": classification,\n        "first_token_gold": first_gold, "first_step_logits_sha256": first_logits_sha,\n        "pulse_fired": 0 if pulse is None else pulse.fired_count, "capture_fired": capture.fired_count,\n        "prefill_forward_count": 1, "decode_forward_count": decode_forwards, "final_cache_length": _cache_length(cache),\n        "requested_delta_norms": None if pulse is None else pulse.requested_delta_norms.tolist(),\n        "actual_delta_norms": None if pulse is None else pulse.actual_delta_norms.tolist(),\n        "elapsed_seconds": elapsed, "prefill_seconds": prefill_seconds, "decode_and_logging_seconds": elapsed-prefill_seconds,\n        "timing_note": \'Synchronized wall time includes capture and top-k logging; not energy telemetry.\',\n        "dtype": \'float32\', "attention": \'sdpa/EFFICIENT_ATTENTION\' if efficient_gpu else str(getattr(model.config, \'_attn_implementation\', None)),\n        "capture_scope": \'last min(4,prompt_length) PROMPT positions; no generated-token activation capture\'}\n    record[\'baseline_kind\'] = \'no_pulse_with_passive_capture\' if arm is None else \'prefill_pulse_with_passive_capture\'\n    return record, arrays\n', 'attention_probe/completion_128.py': '"""Isolated EOS continuation cap; the original 48-token decoder is unchanged.\n\nThe exact existing decoder code runs with a private copy of its globals whose\nsingle MAX_NEW_TOKENS guard is 128. No module global is changed. Its original\n48 default is never used here: the wrapper supplies the total cap explicitly.\n"""\nfrom __future__ import annotations\n\nimport types\n\nfrom . import answer_validation as original\n\nMAX_NEW_TOKENS = 128\n_NAMESPACE = dict(original.__dict__, MAX_NEW_TOKENS=MAX_NEW_TOKENS)\n_DECODER = types.FunctionType(original.run_generation_arm.__code__, _NAMESPACE,\n                              original.run_generation_arm.__name__, original.run_generation_arm.__defaults__,\n                              original.run_generation_arm.__closure__)\n_DECODER.__kwdefaults__ = dict(original.run_generation_arm.__kwdefaults__ or {})\n\n\ndef run_generation_arm(*args, max_new_tokens=MAX_NEW_TOKENS, **kwargs):\n    """Fresh prompt/cache, original prefill pulse, at most 128 total selected tokens."""\n    if type(max_new_tokens) is not int or not 1 <= max_new_tokens <= MAX_NEW_TOKENS:\n        raise ValueError(\'max_new_tokens must be an integer from 1 through 128\')\n    return _DECODER(*args, max_new_tokens=max_new_tokens, **kwargs)\n', 'qwen3_h286_completion_128_runtime.py': '"""Four prespecified EOS-completion checks; keep the completed 48-token run intact."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport time\nimport traceback\nimport types\nimport zipfile\nfrom pathlib import Path\n\nimport qwen3_h286_validation_runtime as previous\n\nbase = previous.base\nSCHEMA = \'qwen3_h286_completion_128_v1\'\nPROOF_SCHEMA = \'qwen3_h286_completion_128_checks_v1\'\nINDICES = (12, 13, 14, 15)\nREQUEST_IDS = tuple(f\'r{i:03d}\' for i in INDICES)\nBASELINE_SHA256 = \'930d772f999b91931023cc1dbbc4bbd98ad68eb27717d6b6fdc6356d1f2e1bd0\'\nSOURCE_ARCHIVE_SHA256 = \'004eae9d743a529aa8e6661c11e892871a41fc6e43745310274a6ae524d432e2\'\nSOURCE_PLAN_SHA256 = \'179f4a37cfd2822a3f4011b7e18c1050cd017bb96484e4dfa270919dcfa2ce4e\'\nMAX_NEW_TOKENS = 128\n\n\ndef generation_module():\n    from attention_probe import completion_128\n    return completion_128\n\n\ndef validation_functions():\n    """Reuse validated code with consistent private globals, never patch old modules."""\n    scope = dict(previous.__dict__, SCHEMA=SCHEMA, PROOF_SCHEMA=PROOF_SCHEMA,\n                 MAX_NEW_TOKENS=MAX_NEW_TOKENS, generation_module=generation_module)\n    for name in (\'verify_proof\', \'technical_checks\', \'validate_generation\', \'validate_payload\',\n                 \'compute_request\', \'verified_request\', \'restore_resume\'):\n        old = getattr(previous, name)\n        new = types.FunctionType(old.__code__, scope, name, old.__defaults__, old.__closure__)\n        new.__kwdefaults__ = dict(old.__kwdefaults__ or {})\n        scope[name] = new\n    return scope\n\n\nFUNCTIONS = validation_functions()\n\n\ndef load_baseline(path):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != BASELINE_SHA256:\n        raise ValueError(\'Exact private 48-token baseline SHA mismatch\')\n    value = json.loads(raw)\n    if (value.get(\'schema\') != \'h286_completion_128_baseline_v1\'\n            or value.get(\'source_archive_sha256\') != SOURCE_ARCHIVE_SHA256\n            or value.get(\'source_plan_sha256\') != SOURCE_PLAN_SHA256\n            or set(value.get(\'requests\', {})) != set(REQUEST_IDS)\n            or hashlib.sha256(base.canonical(value[\'old_plan\'])).hexdigest() != SOURCE_PLAN_SHA256):\n        raise ValueError(\'Baseline source/plan/four original request identities mismatch\')\n    return value\n\n\ndef plan_document(reference, stats, items, baseline):\n    original_plan = previous.plan_document(reference, stats, items)\n    if hashlib.sha256(base.canonical(original_plan)).hexdigest() != SOURCE_PLAN_SHA256:\n        raise ValueError(\'Original scientific sources/references differ from the completed 48-token run\')\n    result = dict(original_plan)\n    result.update(schema=SCHEMA, purpose=\'post-result-selected four-input EOS continuation only\',\n        previous_validation_plan_sha256=SOURCE_PLAN_SHA256, baseline_sha256=BASELINE_SHA256,\n        source_archive_sha256=SOURCE_ARCHIVE_SHA256, request_indices=list(INDICES),\n        helper_sha256=base.sha_file(__file__), completion_module_sha256=base.sha_file(generation_module().__file__),\n        technical_proof_schema=PROOF_SCHEMA,\n        unchanged_scientific_source_sha256=baseline[\'embedded_source\'],\n        queue_budget={\'scope\': \'four requests times five arms times total 128 selected tokens\',\n                      \'wallclock_limit_seconds\': None, \'wallclock_finish_guaranteed\': False},\n        decoder_adapter={\'source\': \'unchanged original decoder code object\',\n                         \'isolated_global_change\': {\'MAX_NEW_TOKENS\': [48, 128]}},\n        generation=dict(original_plan[\'generation\'], max_new_tokens=128,\n            compatibility=\'fresh generation from original prompt; exact old selected-token prefix including old EOS\',\n            numerical_diagnostics=\'logit/top-k/logprob/prefill hashes reported separately; not hard token-prefix conditions\'),\n        inference_scope=\'diagnostic follow-up selected after old results; no independent confirmation\',\n        semantic_review=\'separate human review after EOS; surface classification is not semantic correctness\')\n    result[\'module_sha256\'] = dict(original_plan[\'module_sha256\'],\n                                  **{\'completion_128.py\': result[\'completion_module_sha256\']})\n    return result\n\n\ndef check_original_request(request, prior):\n    old = prior[\'request\']\n    for key in (\'request_id\', \'item_id\', \'pair_id\', \'prompt_ids\', \'formatted_prompt_sha256\',\n                \'answer_ids\', \'pulse_positions\'):\n        if request.get(key) != old.get(key):\n            raise ValueError(\'Original request/tokenization changed: \' + key)\n\n\ndef compare_prefix(record, prior, arrays):\n    """Token/EOS compatibility is hard; numerical bitwise diagnostics are separate."""\n    import numpy as np\n    old = prior[\'generation\']\n    if record.get(\'arm\') != prior[\'arm\'] or record.get(\'seed\') != old[\'seed\']:\n        raise ValueError(\'Arm or original per-request random-control seed changed\')\n    selected = old[\'selected_token_ids\']\n    token_equal = record[\'selected_token_ids\'][:len(selected)] == selected\n    if old[\'stop_reason\'] == \'eos\':\n        eos_equal = record[\'stop_reason\'] == \'eos\' and record[\'selected_token_ids\'] == selected\n    else:\n        eos_equal = len(selected) == 48 and len(record[\'selected_token_ids\']) >= 48\n    old_steps, new_steps = old[\'steps\'], record[\'steps\'][:len(selected)]\n    steps_equal = old_steps == new_steps\n    logprob_differences = [abs(a[\'selected_token_logprob\'] - b[\'selected_token_logprob\'])\n                           for a, b in zip(old_steps, new_steps)]\n    hashes = {key: hashlib.sha256(np.asarray(value).tobytes(order=\'C\')).hexdigest()\n              for key, value in arrays.items() if key.startswith(\'gen_\')}\n    expected_hashes = prior[\'gen_array_sha256\']\n    eos_protocol_equal = (record.get(\'eos_token_ids\') == old.get(\'eos_token_ids\')\n                          and record.get(\'eos_source\') == old.get(\'eos_source\'))\n    return {\'compatible\': bool(token_equal and eos_equal and eos_protocol_equal),\n            \'eos_protocol_equal\': eos_protocol_equal, \'tokens_equal\': token_equal,\n            \'old_eos_or_cap_preserved\': eos_equal, \'old_selected_tokens\': len(selected),\n            \'old_stop_reason\': old[\'stop_reason\'], \'first_logits_exact\': record[\'first_step_logits_sha256\'] == old[\'first_step_logits_sha256\'],\n            \'prefix_steps_logprobs_topk_exact\': steps_equal,\n            \'max_abs_selected_logprob_difference\': max(logprob_differences, default=0.),\n            \'prefill_arrays_exact\': hashes == expected_hashes,\n            \'prefill_array_shapes_exact\': {key: list(value.shape) for key, value in arrays.items()\n                                           if key.startswith(\'gen_\')} == prior[\'gen_array_shapes\'],\n            \'prefill_array_sha256\': hashes,\n            \'requested_delta_norms_exact\': record[\'requested_delta_norms\'] == old[\'requested_delta_norms\'],\n            \'actual_delta_norms_exact\': record[\'actual_delta_norms\'] == old[\'actual_delta_norms\'],\n            \'numerical_diagnostics_are_additional_hard_conditions\': False}\n\n\ndef compare_result(request, rows, arrays, baseline):\n    prior = baseline[\'requests\'][request[\'request_id\']]\n    check_original_request(request, prior)\n    checks = []\n    for row, old, values in zip(rows, prior[\'arms\'], arrays, strict=True):\n        check = compare_prefix(row[\'generation\'], old, values)\n        row[\'continuation_48_check\'] = check\n        row[\'completion\'] = {\'eos_finished\': row[\'generation\'][\'stop_reason\'] == \'eos\',\n                             \'truncated_at_128\': row[\'generation\'][\'truncated\']}\n        row[\'format\'] = row[\'generation\'][\'classification_details\']\n        row[\'semantic_answer\'] = {\'status\': \'requires_separate_review\',\n            \'target\': request[\'answer\'], \'reviewable_complete_output\': not row[\'generation\'][\'truncated\'],\n            \'note\': \'Do not infer correctness from truncation, gold mentions, or noncanonical formatting.\'}\n        checks.append(check)\n    return checks\n\n\ndef verified_request(output, request_id, plan_hash, count, baseline):\n    import numpy as np\n    record = FUNCTIONS[\'verified_request\'](output, request_id, plan_hash, count)\n    if record is None:\n        return None\n    with np.load(Path(output)/\'requests\'/request_id/\'arrays.npz\', allow_pickle=False) as saved:\n        arrays = [{key: saved[key][i] for key in saved.files} for i in range(5)]\n    original_checks = [row.get(\'continuation_48_check\') for row in record[\'arms\']]\n    checks = compare_result(record[\'request\'], record[\'arms\'], arrays, baseline)\n    if checks != original_checks:\n        raise ValueError(\'Stored prefix compatibility diagnostics changed\')\n    return record\n\n\ndef archive(output):\n    target = Path(output)/\'h286_completion_128_results.zip\'\n    temporary = target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as packed:\n        for path in sorted(Path(output).rglob(\'*\')):\n            if path.is_file() and path.suffix not in (\'.zip\', \'.tmp\') and not any(\n                    part.startswith(\'.\') for part in path.relative_to(output).parts):\n                packed.write(path, path.relative_to(output))\n    os.replace(temporary, target)\n\n\ndef run(hn_set, policy, refw, items_path, baseline_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir); output.mkdir(parents=True, exist_ok=True)\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'scope\': SCHEMA, \'completed_requests\': 0,\n              \'confirmatory_eligible\': False}\n    metrics = {\'weights_loaded\': False, \'pretrained_model_run\': False,\n               \'wrapper_attempt_id\': os.environ.get(\'H286_WRAPPER_ATTEMPT\'), \'checks\': {},\n               \'semantic_scoring\': \'separate review required; no automatic semantic-correctness rescue\'}\n    started = time.monotonic()\n    def save(extra=None):\n        if extra: status.update(extra)\n        base.atomic_json(output/\'status.json\', status); base.atomic_json(output/\'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items = base.load_inputs(hn_set, policy, refw, items_path)\n        baseline = load_baseline(baseline_path)\n        plan = plan_document(reference, stats, items, baseline)\n        digest = hashlib.sha256(base.canonical(plan)).hexdigest()\n        path = output/\'run_plan.json\'\n        if path.exists() and path.read_bytes() != base.canonical(plan)+b\'\\n\':\n            raise ValueError(\'Existing completion output has another plan; no mixing or overwrite\')\n        if not path.exists(): base.atomic_json(path, plan)\n        base.atomic_json(output/\'prior_48_provenance.json\', {\'baseline_sha256\': BASELINE_SHA256,\n            \'source_archive_sha256\': SOURCE_ARCHIVE_SHA256, \'source_plan_sha256\': SOURCE_PLAN_SHA256,\n            \'requests\': {rid: {key: value[key] for key in (\'source_result_sha256\', \'source_arrays_sha256\')}\n                         for rid, value in baseline[\'requests\'].items()}})\n        if resume_zip:\n            with zipfile.ZipFile(resume_zip) as packed:\n                ids = {Path(name).parts[1] for name in packed.namelist() if name.startswith(\'requests/\') and len(Path(name).parts)>2}\n                if not ids.issubset(REQUEST_IDS): raise ValueError(\'Resume contains requests outside the four-input continuation\')\n            FUNCTIONS[\'restore_resume\'](resume_zip, output, digest, len(reference.coordinates))\n        done = {rid: verified_request(output, rid, digest, len(reference.coordinates), baseline) for rid in REQUEST_IDS}\n        status[\'completed_requests\'] = sum(row is not None for row in done.values())\n        if not all(done.values()):\n            status[\'stage\'] = \'hardware_and_original_model_load\'; save()\n            def loaded(hardware):\n                metrics.update(weights_loaded=True, hardware=hardware); save()\n            engine = previous.initialize_engine(reference, Path(cache_dir or Path.home()/\'.cache/h286_pilot\'), on_loaded=loaded)\n            model, tokenizer = engine[\'model\'], engine[\'tokenizer\']\n            import torch\n            first = next(i for i in INDICES if done[f\'r{i:03d}\'] is None)\n            request = base.prepare_request(tokenizer, first, items[first])\n            check_original_request(request, baseline[\'requests\'][request[\'request_id\']])\n            with torch.inference_mode(), previous.attention_context(torch, True):\n                ids = request[\'ids\'].to(next(model.parameters()).device)\n                model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n            metrics[\'pretrained_model_run\'] = True\n            status[\'stage\'] = \'unchanged_controls_with_cap128\'; save()\n            proof = FUNCTIONS[\'technical_checks\'](model, tokenizer, request, reference, stats, digest,\n                engine[\'hardware\'], weight_norms=engine[\'weight_norms\'])\n            metrics[\'checks\'] = proof[\'checks\']\n            frozen = base.parameter_evidence(model)\n            for index in INDICES:\n                rid = f\'r{index:03d}\'\n                if done[rid] is not None: continue\n                request = base.prepare_request(tokenizer, index, items[index])\n                check_original_request(request, baseline[\'requests\'][rid])\n                status.update(stage=\'four_input_completion\', current_request=rid); save()\n                print(f\'Request {rid}: originaler Prompt, fünf Arme, Gesamtkappe128.\', flush=True)\n                cpu_rng, cuda_rng = torch.random.get_rng_state().clone(), torch.cuda.get_rng_state().clone()\n                rows, arrays = FUNCTIONS[\'compute_request\'](model, tokenizer, request, reference, stats,\n                    weight_norms=engine[\'weight_norms\'])\n                checks = compare_result(request, rows, arrays, baseline)\n                if (base.parameter_evidence(model) != frozen or not torch.equal(cpu_rng, torch.random.get_rng_state())\n                        or not torch.equal(cuda_rng, torch.cuda.get_rng_state())):\n                    raise ValueError(\'Original frozen parameter/global RNG evidence changed\')\n                done[rid] = base.write_request(output, request, rows, arrays, digest, engine[\'hardware\'], proof)\n                status[\'completed_requests\'] = sum(row is not None for row in done.values()); save()\n                print(f\'Gespeichert {status["completed_requests"]}/4; Präfix passend: {sum(c["compatible"] for c in checks)}/5.\', flush=True)\n        all_checks = [arm[\'continuation_48_check\'] for row in done.values() for arm in row[\'arms\']]\n        metrics[\'prefix_compatibility_passed\'] = all(check[\'compatible\'] for check in all_checks)\n        metrics[\'prefix_checks\'] = all_checks\n        metrics[\'retained_request_proofs\'] = [row[\'technical_checks\'] for row in done.values()]\n        status.update(state=\'complete\' if metrics[\'prefix_compatibility_passed\'] else \'failed\',\n                      stage=\'complete\' if metrics[\'prefix_compatibility_passed\'] else \'prefix_compatibility_failure\')\n        if not metrics[\'prefix_compatibility_passed\']:\n            status[\'error\'] = \'Old selected-token prefix/EOS differs; longer text is not an established compatible continuation.\'\n    except BaseException as error:  # noqa: BLE001 -- retain diagnostics after interruption.\n        status.update(state=\'failed\', error_type=type(error).__name__, error=str(error))\n        (output/\'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \'+str(error), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic()-started\n        save(); archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in (\'hn-set\', \'policy\', \'refw\', \'items\', \'baseline\', \'out-dir\'):\n        parser.add_argument(\'--\'+name, required=True)\n    parser.add_argument(\'--cache-dir\'); parser.add_argument(\'--resume-zip\')\n    args = parser.parse_args()\n    result = run(args.hn_set, args.policy, args.refw, args.items, args.baseline, args.out_dir, args.cache_dir, args.resume_zip)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\']==\'complete\' else 1)\n', 'attention_probe/group_restoration.py': '"""One-prefill signed gated-MLP interventions for the frozen native Qwen3.\n\nA channel is a column of ``SiLU(gate_proj(x)) * up_proj(x)`` immediately before\n``down_proj``. It is neither an attention head nor a standardized CETT score.\nPositions are absolute physical columns of a full, unfilled prompt. The caller\nmust supply only the intended last-prompt column for the group experiment.\n\nEvery context removes all its own hooks when the first fresh prefill succeeds.\nOn a failed forward the model-post hooks are removed at context exit (PyTorch\'s\nexception handler iterates that hook dictionary directly). Cached decoding\ninside the still-open successful context therefore has no intervention/collector\nhooks. This module performs no model loading,\ncoordinate selection, generation, device transfer of models, or parameter edits.\n"""\nfrom __future__ import annotations\n\nimport inspect\nimport random\nfrom collections.abc import Iterable, Sequence\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom .interventions import _cache_length, _integers, _layers\n\n\ndef _integer(value, name: str, *, positive: bool = False) -> int:\n    result = _integers((value,), name)[0]\n    if positive and result == 0:\n        raise ValueError(f"{name} must be positive")\n    return result\n\n\ndef _cpu_copy(value: torch.Tensor) -> torch.Tensor:\n    return value.detach().to(device="cpu", copy=True)\n\n\ndef _source(value, name: str) -> torch.Tensor:\n    if not isinstance(value, torch.Tensor) or value.ndim != 3:\n        raise ValueError(f"{name} must be a CPU float32 tensor with shape [B,P,C]")\n    if value.device.type != "cpu" or value.dtype != torch.float32:\n        raise ValueError(f"{name} must be CPU float32; implicit casts are forbidden")\n    if value.requires_grad or value.grad_fn is not None:\n        raise ValueError(f"{name} must be detached, without a live autograd graph")\n    if any(size <= 0 for size in value.shape) or not bool(torch.isfinite(value).all()):\n        raise ValueError(f"{name} must have nonempty axes and finite values")\n    return value.detach().clone()\n\n\ndef _native_model(model):\n    cls = type(model)\n    if (cls.__name__ != "Qwen3ForCausalLM"\n            or cls.__module__ != "transformers.models.qwen3.modeling_qwen3"\n            or getattr(getattr(model, "config", None), "model_type", None) != "qwen3"):\n        raise ValueError("Group restoration requires native Hugging Face Qwen3ForCausalLM")\n    if model.training:\n        raise ValueError("Group restoration requires model.eval(); mode is never changed")\n    params = tuple(model.parameters())\n    if not params or any(p.requires_grad for p in params):\n        raise ValueError("Group restoration requires frozen model parameters")\n    if any(p.is_floating_point() and p.dtype != torch.float32 for p in params):\n        raise ValueError("Group restoration requires native FP32 parameters")\n    devices = {p.device for p in params}\n    if len(devices) != 1 or next(iter(devices)).type not in ("cpu", "cuda"):\n        raise ValueError("Group restoration requires one CPU/CUDA device; no offload or meta parameters")\n    return next(iter(devices))\n\n\nclass _AutoPrefill:\n    """One fresh forward; remove own hooks at its outer model return."""\n\n    def __init__(self, model, positions: Sequence[int] | None):\n        self.model = model\n        self.device = _native_model(model)\n        self.positions = None if positions is None else _integers(positions, "positions")\n        self.layers = _layers(model)\n        self._handles = []\n        self._post_handles = []\n        self._normal_post_ran = False\n        self._used = False\n        self._claimed = False\n        self._window = None\n        self._positions = None\n        self.full_passes = 0\n        self.completed_count = 0\n        self.skipped_cached_passes = 0\n        self.removed_before_decode = False\n\n    def _model_pre(self, module, args, kwargs):\n        self._normal_post_ran = False\n        self._window = None\n        if self._claimed:\n            return\n        # Bind positional cache/mask arguments too; HF forwards also accept kwargs.\n        try:\n            bound = inspect.signature(module.forward).bind_partial(*args, **kwargs).arguments\n        except (TypeError, ValueError):\n            bound = {}\n        fields = dict(kwargs)\n        fields.update({k: v for k, v in bound.items() if k not in ("args", "kwargs")})\n        length = _cache_length(fields.get("past_key_values", fields.get("past_key_value")))\n        offset = fields.get("cache_position")\n        nonzero_offset = (offset is not None and torch.as_tensor(offset).numel() > 0\n                          and int(torch.as_tensor(offset).reshape(-1)[0]) != 0)\n        if length is None or length > 0 or nonzero_offset:\n            self.skipped_cached_passes += 1\n            return\n        value = fields.get("input_ids")\n        if value is None:\n            value = fields.get("inputs_embeds")\n        if value is None and args:\n            value = args[0]\n        if not isinstance(value, torch.Tensor) or value.ndim not in (2, 3):\n            raise ValueError("Cannot identify complete prompt input")\n        batch, tokens = map(int, value.shape[:2])\n        if batch <= 0 or tokens <= 0:\n            raise ValueError("Prompt must have nonempty batch and sequence")\n        positions = (tokens - 1,) if self.positions is None else self.positions\n        if max(positions) >= tokens:\n            raise ValueError("Selected position is outside the full prompt")\n        mask = fields.get("attention_mask")\n        if mask is not None:\n            if not isinstance(mask, torch.Tensor) or tuple(mask.shape) != (batch, tokens):\n                raise ValueError("Expected a two-dimensional full-prompt attention mask")\n            if not bool((mask[:, list(positions)] == 1).all()):\n                raise ValueError("Selected position must be a real, unmasked token in every row")\n        # Recheck mode/parameter metadata at the actual forward (not only construction).\n        if _native_model(module) != self.device:\n            raise ValueError("Model device changed after hook construction")\n        if torch.is_autocast_enabled(self.device.type):\n            raise ValueError("Group restoration requires autocast disabled")\n        if self.device.type == "cuda" and (torch.backends.cuda.matmul.allow_tf32\n                                           or torch.backends.cudnn.allow_tf32):\n            raise ValueError("Group restoration requires CUDA matmul and cuDNN TF32 disabled")\n        self._positions = positions\n        self._window = (batch, tokens)\n        self._claimed = True\n        self.full_passes += 1\n        self._prepare(batch, positions)\n\n    def _prepare(self, batch, positions):\n        pass\n\n    def _model_post(self, module, args, output):\n        self._normal_post_ran = True\n        if self._window is None:\n            return  # A cached call before the fresh prefill is never claimed.\n        try:\n            self._finish(output)\n            self.completed_count += 1\n        finally:\n            self.close()\n        self.removed_before_decode = True\n        return\n\n    def _model_error(self, module, args, output):\n        # On success _model_post already removed all handles. On failure it did\n        # not run, including when another post-hook failed with non-None output.\n        # PyTorch\'s exception path iterates model._forward_hooks directly, so\n        # retain our model-post handles until __exit__ rather than mutate that\n        # OrderedDict and replace the original error. Other hooks go now.\n        if self._normal_post_ran:\n            return\n        self._remove_handles(keep_post=True)\n        self._window = None\n        return\n\n    def _install(self):\n        raise NotImplementedError\n\n    def _finish(self, output):\n        raise NotImplementedError\n\n    def validate_fired(self):\n        if self.full_passes != 1 or self.completed_count != 1:\n            raise RuntimeError("Expected exactly one successful fresh prefill")\n\n    def __enter__(self):\n        if self._used:\n            raise RuntimeError("Use a fresh group hook context for each prefill")\n        self._used = True\n        try:\n            self._handles.append(self.model.register_forward_pre_hook(self._model_pre, with_kwargs=True))\n            self._post_handles = [\n                self.model.register_forward_hook(self._model_post),\n                self.model.register_forward_hook(self._model_error, always_call=True),\n            ]\n            self._handles.extend(self._post_handles)\n            self._install()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def _remove_handles(self, *, keep_post=False):\n        for handle in reversed(self._handles):\n            if not keep_post or handle not in self._post_handles:\n                handle.remove()\n        self._handles = [h for h in self._handles if keep_post and h in self._post_handles]\n\n    def close(self):\n        self._remove_handles()\n        self._window = None\n\n    def __exit__(self, kind, error, tb):\n        self.close()\n        if kind is None:\n            self.validate_fired()\n        return False\n\n    def _z(self, args, width):\n        if not args or not isinstance(args[0], torch.Tensor):\n            raise ValueError("Native down_proj must receive a positional tensor")\n        z = args[0]\n        if tuple(z.shape) != (*self._window, width) or z.dtype != torch.float32:\n            raise ValueError("Unexpected native FP32 gated-MLP input shape/dtype")\n        if z.device != self.device:\n            raise ValueError("Gated-MLP input is on the wrong device")\n        return z\n\n\nclass GroupZPatch(_AutoPrefill):\n    """Replace or add signed raw-z values at exactly the selected cells once.\n\n    ``donor_values`` and ``delta`` are mutually exclusive detached CPU FP32\n    tensors, with exact shape [batch, len(positions), len(channels)]; no broadcast\n    or dtype conversion is permitted. Neither supplied means an exact no-op.\n    Replacement assigns donor FP32 values directly (not via subtract/add).\n\n    ``record`` stores detached CPU before/after values, per-[B,P] L2 norms, and\n    optionally the projected delta vector. Projection uses the selected columns\n    of native W_down; it describes linear geometry, not measured output changes,\n    compute saved, memory capacity, or energy. Norms are evaluated in float64.\n    """\n\n    def __init__(self, model, block: int, positions: Sequence[int], channels: Sequence[int], *,\n                 donor_values=None, delta=None, record_projected_vector: bool = True):\n        super().__init__(model, positions)\n        self.block = _integer(block, "block")\n        self.channels = _integers(channels, "channels")\n        if self.block >= len(self.layers):\n            raise ValueError("block is outside the native decoder")\n        self.down_proj = self.layers[self.block].mlp.down_proj\n        if type(self.down_proj) is not nn.Linear:\n            raise ValueError("Expected the native dense down_proj Linear")\n        if max(self.channels) >= self.down_proj.in_features:\n            raise ValueError("channel is outside the gated-MLP width")\n        if donor_values is not None and delta is not None:\n            raise ValueError("Supply donor_values OR delta, never both")\n        if not isinstance(record_projected_vector, bool):\n            raise ValueError("record_projected_vector must be boolean")  # noqa: TRY004 -- consistent invalid-configuration API.\n        self.mode = "replace" if donor_values is not None else "add" if delta is not None else "identity"\n        self._source = _source(donor_values, "donor_values") if donor_values is not None else (\n            _source(delta, "delta") if delta is not None else None)\n        self.record_projected_vector = record_projected_vector\n        self.fired_count = 0\n        self.record = {}\n\n    def _prepare(self, batch, positions):\n        expected = (batch, len(positions), len(self.channels))\n        if self._source is not None and tuple(self._source.shape) != expected:\n            raise ValueError(f"Patch source must have exact shape {expected}; no broadcasting")\n\n    def _install(self):\n        # prepend guarantees passive CoordinateCapture prehooks observe patched z,\n        # even if they were installed before this patch. Other mutators must not\n        # target the same z site in the same forward.\n        self._handles.append(self.down_proj.register_forward_pre_hook(self._patch, prepend=True))\n\n    def _patch(self, module, args):\n        if self._window is None:\n            return None\n        if self.fired_count:\n            raise RuntimeError("Selected down_proj fired more than once during one prefill")\n        z = self._z(args, module.in_features)\n        p = torch.tensor(self._positions, device=z.device, dtype=torch.long)\n        c = torch.tensor(self.channels, device=z.device, dtype=torch.long)\n        with torch.no_grad():\n            before = z[:, p[:, None], c[None, :]]\n            if not bool(torch.isfinite(before).all()):\n                raise ValueError("Selected native raw-z contains nonfinite values")\n            if self._source is None:\n                after = before\n            else:\n                source = self._source.to(device=z.device)\n                after = source if self.mode == "replace" else before + source\n            if not bool(torch.isfinite(after).all()):\n                raise ValueError("Patch would produce nonfinite raw-z")\n            actual_delta = after - before\n            projected = F.linear(actual_delta, module.weight[:, c])\n            if not bool(torch.isfinite(projected).all()):\n                raise ValueError("Projected patch delta is nonfinite")\n            self.record = {\n                "receiver_before": _cpu_copy(before),\n                "receiver_after": _cpu_copy(after),\n                "raw_delta_norm": actual_delta.double().norm(dim=-1).cpu(),\n                "projected_delta_norm": projected.double().norm(dim=-1).cpu(),\n            }\n            if self.record_projected_vector:\n                self.record["projected_delta_vector"] = _cpu_copy(projected)\n            self.fired_count += 1\n            if torch.equal(before, after):\n                return None  # Identity preserves the original input tensor exactly.\n            changed = z.clone()\n            changed[:, p[:, None], c[None, :]] = after\n            return (changed, *args[1:])\n\n    def _finish(self, output):\n        if self.fired_count != 1:\n            raise RuntimeError("Group patch must fire exactly once in the fresh prefill")\n\n    def validate_fired(self):\n        super().validate_fired()\n        if self.fired_count != 1:\n            raise RuntimeError("Group patch did not fire exactly once")\n\n\nclass ProbeCollector(_AutoPrefill):\n    """Capture selected full raw-z rows and the first next-token logits once.\n\n    By default only the last physical prompt column is copied. Set explicit\n    positions only for a separately specified diagnostic. z_by_block[block] is\n    detached CPU FP32 [B,P,intermediate_size]; first_logits is detached CPU FP32\n    [B,vocab_size]. Results remain available after context exit. This collector\n    works both with use_cache=True and with a plain use_cache=False prefill.\n    """\n\n    def __init__(self, model, blocks: Sequence[int] = (7, 14), positions=None):\n        super().__init__(model, positions)\n        self.blocks = _integers(blocks, "blocks")\n        if max(self.blocks) >= len(self.layers):\n            raise ValueError("block is outside the native decoder")\n        self.z_by_block = {}\n        self.first_logits = None\n        self.fired_count = 0\n\n    def _install(self):\n        for block in self.blocks:\n            down = self.layers[block].mlp.down_proj\n            if type(down) is not nn.Linear:\n                raise ValueError("Expected native dense down_proj Linear")\n            self._handles.append(down.register_forward_pre_hook(\n                lambda module, args, block=block: self._capture(block, module, args)))\n\n    def _capture(self, block, module, args):\n        if self._window is None:\n            return\n        if block in self.z_by_block:\n            raise RuntimeError("Probe down_proj fired more than once during one prefill")\n        z = self._z(args, module.in_features)\n        chosen = z[:, list(self._positions), :]\n        if not bool(torch.isfinite(chosen).all()):\n            raise ValueError("Probe raw-z contains nonfinite values")\n        self.z_by_block[block] = _cpu_copy(chosen)\n        return\n\n    def _finish(self, output):\n        if set(self.z_by_block) != set(self.blocks):\n            raise RuntimeError("Not all requested raw-z blocks were captured")\n        logits = getattr(output, "logits", None)\n        if logits is None and isinstance(output, tuple) and output:\n            logits = output[0]\n        batch = self._window[0]\n        if (not isinstance(logits, torch.Tensor) or logits.ndim != 3\n                or logits.shape[0] != batch or logits.shape[1] == 0\n                or logits.dtype != torch.float32 or logits.device != self.device):\n            raise ValueError("Expected native FP32 causal-LM logits [B,T,V]")\n        last = logits[:, -1, :]\n        if not bool(torch.isfinite(last).all()):\n            raise ValueError("First next-token logits contain nonfinite values")\n        self.first_logits = _cpu_copy(last)\n        self.fired_count += 1\n\n    def validate_fired(self):\n        super().validate_fired()\n        if self.fired_count != 1:\n            raise RuntimeError("Probe did not complete exactly once")\n\n\ndef channels_in_block(coordinates: Iterable[Sequence[int]], block: int) -> tuple[int, ...]:\n    """Select a block\'s channels while preserving the reference coordinate order."""\n    block = _integer(block, "block")\n    pairs = tuple(tuple(pair) for pair in coordinates)\n    for pair in pairs:\n        if len(pair) != 2:\n            raise ValueError("coordinates must be (block, channel) integer pairs")\n        _integers((pair[0],), "coordinate block")\n        _integers((pair[1],), "coordinate channel")\n    if len(set(pairs)) != len(pairs):\n        raise ValueError("coordinates must be unique")\n    return tuple(int(channel) for layer, channel in pairs if layer == block)\n\n\ndef sample_control_masks(width: int, excluded: Iterable[int], count: int, *,\n                         seeds: Sequence[int]) -> tuple[tuple[int, ...], ...]:\n    """Deterministic masks outside H, with a separate local Python RNG per seed.\n\n    Channels are sorted in each mask. Masks can overlap across seeds, but exact\n    duplicate masks are rejected rather than silently reseeded. Seeds belong in\n    the frozen runner plan, and must not be chosen based on measured outcomes.\n    """\n    width = _integer(width, "width", positive=True)\n    count = _integer(count, "count", positive=True)\n    excluded = _integers(excluded, "excluded", nonempty=False)\n    seeds = _integers(seeds, "seeds")\n    if excluded and max(excluded) >= width:\n        raise ValueError("excluded channel is outside width")\n    allowed = sorted(set(range(width)).difference(excluded))\n    if count > len(allowed):\n        raise ValueError("Not enough outside-H channels for the matched mask")\n    masks = tuple(tuple(sorted(random.Random(seed).sample(allowed, count))) for seed in seeds)\n    if len(set(masks)) != len(masks):\n        raise ValueError("Seeds produced duplicate complete masks; specify distinct frozen seeds")\n    return masks\n\n\ndef random_matched_delta(delta: torch.Tensor, *, seed: int) -> torch.Tensor:\n    """Gaussian direction with per-[B,P] raw-delta norm, then one FP32 cast.\n\n    A local CPU torch.Generator leaves global RNG state unchanged. Matching is\n    performed in float64 before the final FP32 cast; callers must report actual\n    norms and W_down projections after native FP32 addition. Zero target norm\n    produces exactly zero (not an invented random signal). No direction is\n    resampled to improve its projected alignment or effect on model outputs.\n    """\n    source = _source(delta, "delta")\n    seed = _integer(seed, "seed")\n    target = source.double().norm(dim=-1, keepdim=True)\n    if not bool((target > 0).any()):\n        return torch.zeros_like(source)\n    generator = torch.Generator(device="cpu").manual_seed(seed)\n    direction = torch.randn(source.shape, dtype=torch.float64, generator=generator)\n    length = direction.norm(dim=-1, keepdim=True)\n    if bool(((target > 0) & (length == 0)).any()):\n        raise ValueError("Degenerate zero Gaussian direction; do not silently reseed")\n    direction = direction / length.clamp_min(torch.finfo(torch.float64).tiny)\n    return (direction * target).float()\n', 'attention_probe/group_probe_items.json': '[\n  {\n    "id": "p07_correct_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: green. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application",\n    "request_id": "r012",\n    "variant": "original_correct_draft",\n    "source_item_id": "p07_correct_draft",\n    "source_request_id": "r012",\n    "family": "p07_wug_color",\n    "semantic_alternative": "brown",\n    "output_instruction": "color_only",\n    "primary_original": true\n  },\n  {\n    "id": "p07_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: brown. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application",\n    "request_id": "r013",\n    "variant": "original_wrong_draft",\n    "source_item_id": "p07_wrong_draft",\n    "source_request_id": "r013",\n    "family": "p07_wug_color",\n    "semantic_alternative": "brown",\n    "output_instruction": "color_only",\n    "primary_original": true\n  },\n  {\n    "id": "p08_correct_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: B. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic",\n    "request_id": "r014",\n    "variant": "original_correct_draft",\n    "source_item_id": "p08_correct_draft",\n    "source_request_id": "r014",\n    "family": "p08_exclusive_switches",\n    "semantic_alternative": "A",\n    "output_instruction": "A_or_B_only",\n    "primary_original": true\n  },\n  {\n    "id": "p08_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: A. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic",\n    "request_id": "r015",\n    "variant": "original_wrong_draft",\n    "source_item_id": "p08_wrong_draft",\n    "source_request_id": "r015",\n    "family": "p08_exclusive_switches",\n    "semantic_alternative": "A",\n    "output_instruction": "A_or_B_only",\n    "primary_original": true\n  },\n  {\n    "id": "p07_no_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application",\n    "request_id": "r100",\n    "variant": "no_draft",\n    "source_item_id": "p07_wrong_draft",\n    "source_request_id": "r013",\n    "family": "p07_wug_color",\n    "semantic_alternative": "brown",\n    "output_instruction": "color_only",\n    "primary_original": false\n  },\n  {\n    "id": "p07_untrusted_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Untrusted draft (may be wrong): brown. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application",\n    "request_id": "r101",\n    "variant": "explicitly_untrusted_wrong_draft",\n    "source_item_id": "p07_wrong_draft",\n    "source_request_id": "r013",\n    "family": "p07_wug_color",\n    "semantic_alternative": "brown",\n    "output_instruction": "color_only",\n    "primary_original": false\n  },\n  {\n    "id": "p07_explain_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: brown. Explain briefly which color Pip is.",\n    "target": "green",\n    "category": "rule_application",\n    "request_id": "r102",\n    "variant": "explain_wrong_draft",\n    "source_item_id": "p07_wrong_draft",\n    "source_request_id": "r013",\n    "family": "p07_wug_color",\n    "semantic_alternative": "brown",\n    "output_instruction": "brief_explanation",\n    "primary_original": false\n  },\n  {\n    "id": "p08_no_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic",\n    "request_id": "r103",\n    "variant": "no_draft",\n    "source_item_id": "p08_wrong_draft",\n    "source_request_id": "r015",\n    "family": "p08_exclusive_switches",\n    "semantic_alternative": "A",\n    "output_instruction": "A_or_B_only",\n    "primary_original": false\n  },\n  {\n    "id": "p08_untrusted_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Untrusted draft (may be wrong): A. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic",\n    "request_id": "r104",\n    "variant": "explicitly_untrusted_wrong_draft",\n    "source_item_id": "p08_wrong_draft",\n    "source_request_id": "r015",\n    "family": "p08_exclusive_switches",\n    "semantic_alternative": "A",\n    "output_instruction": "A_or_B_only",\n    "primary_original": false\n  },\n  {\n    "id": "p08_explain_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: A. Which is on? Explain briefly.",\n    "target": "B",\n    "category": "exclusive_logic",\n    "request_id": "r105",\n    "variant": "explain_wrong_draft",\n    "source_item_id": "p08_wrong_draft",\n    "source_request_id": "r015",\n    "family": "p08_exclusive_switches",\n    "semantic_alternative": "A",\n    "output_instruction": "brief_explanation",\n    "primary_original": false\n  }\n]\n', 'qwen3_h286_group_runtime.py': '"""Prespecified local H286 restoration/transfer probe; no training or adaptive arms.\n\nThree fresh same-prompt baselines provide donors. Every input then receives the\nsame 32 local group tests before any prespecified generation is selected. Only\nthree full-vocabulary baseline vectors are saved; all other KLs use those full\nvectors online. Completion/format remain separate from semantic correctness.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\n\nimport qwen3_h286_completion_128_runtime as completion\n\nprevious = completion.previous\nbase = previous.base\nSCHEMA = \'qwen3_h286_group_restoration_v1\'\nCONTINUITY_SHA256 = \'111d7213f76374a459c38ca17f4701eb9228693708ee1657a7e25f987c65ddc1\'\nGROUP_ITEMS_SHA256 = \'0bf76a995436092728cacc46e1be78c92b8e5416b66402324687b2a796b8ec88\'\nBLOCKS = (7, 14)\nCOUNTS = {7: 9, 14: 25}\nORIGINAL_IDS = (\'r012\', \'r013\', \'r014\', \'r015\')\nREQUEST_IDS = ORIGINAL_IDS + tuple(f\'r{i:03d}\' for i in range(100, 106))\nBASELINE_ARMS = (base.ARMS[0], base.ARMS[2], base.ARMS[4])\nBASELINE_NAMES = tuple(arm[\'name\'] for arm in BASELINE_ARMS)\nPULSE_NAME = {7: \'other_block_108\', 14: \'soft_108\'}\nARRAY_CONTRACT = {\'vocab_size\': 151936, \'hidden_size\': 2048, \'mlp_width\': 6144, \'n_coordinates\': 286}\nPROOF_CHECKS = (\'native_clean_logits\', \'identity_logits\', \'prefill_counts\', \'hook_cleanup\',\n                \'generation_prefill_logits\', \'frozen_parameters\', \'global_rng\', \'fresh_same_prompt_donors\')\nMAIN_GROUP_NAMES = tuple(f\'b{block}_{direction}\' for block in BLOCKS for direction in (\'restore\', \'transfer\'))\n\n\ndef source_path(name):\n    root = Path(base.__file__).parent\n    path = root/name\n    if not path.exists():\n        path = root.parent/name\n    return path\n\n\ndef load_inputs(hn, policy, refw, items_path, continuity_path):\n    reference, stats, original_items = base.load_inputs(hn, policy, refw, source_path(\'attention_probe/pilot_items.json\'))\n    raw = Path(items_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != GROUP_ITEMS_SHA256:\n        raise ValueError(\'Exact prespecified ten group-probe inputs required\')\n    items = json.loads(raw)\n    if len(items) != 10 or tuple(item[\'request_id\'] for item in items) != REQUEST_IDS:\n        raise ValueError(\'Group inputs must keep original request IDs and fixed variant IDs\')\n    for item, old in zip(items[:4], original_items[12:16], strict=True):\n        if any(item[key] != old[key] for key in (\'id\', \'pair_id\', \'prompt\', \'target\', \'category\')):\n            raise ValueError(\'Original group-probe prompt/target changed\')\n    raw = Path(continuity_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != CONTINUITY_SHA256:\n        raise ValueError(\'Exact private 128-token continuity reference SHA mismatch\')\n    continuity = json.loads(raw)\n    if (continuity.get(\'schema\') != \'h286_group_continuity_v1\'\n            or set(continuity.get(\'requests\', {})) != set(ORIGINAL_IDS)\n            or continuity.get(\'model\') != reference.model.to_dict()):\n        raise ValueError(\'Continuity model/request identity mismatch\')\n    for name, expected in continuity[\'source_module_sha256\'].items():\n        if base.sha_file(source_path(name)) != expected:\n            raise ValueError(\'Original scientific source changed: \'+name)\n    return reference, stats, items, continuity\n\n\ndef make_controls(coordinates, width):\n    """One fixed mask/direction plan, independent of prompts, deltas and outcomes."""\n    result = {}\n    for block in BLOCKS:\n        group = [channel for layer, channel in coordinates if layer == block]\n        if len(group) != COUNTS[block] or len(set(group)) != len(group):\n            raise ValueError(\'Expected fixed H286 group cardinality at block \'+str(block))\n        outside = sorted(set(range(width))-set(group))\n        if len(outside) < len(group):\n            raise ValueError(\'Not enough outside-H channels for fixed size-matched masks\')\n        masks, within_units, mask_seeds, direction_seeds = [], [], [], []\n        for index in range(3):\n            mask_seed = base.SEED + 10000*block + 100*index\n            mask = sorted(random.Random(mask_seed).sample(outside, len(group)))\n            if mask in masks:\n                raise ValueError(\'Duplicate fixed outside-H mask; no resampling\')\n            masks.append(mask); mask_seeds.append(mask_seed)\n            direction_seed = mask_seed+53; direction_seeds.append(direction_seed)\n            rng = random.Random(direction_seed)\n            direction = [rng.gauss(0., 1.) for _ in group]\n            norm = math.sqrt(sum(value*value for value in direction))\n            within_units.append([value/norm for value in direction] if norm > 0 else None)\n        result[block] = {\'group\': group, \'outside_masks\': masks, \'within_units\': within_units,\n                         \'outside_mask_seeds\': mask_seeds, \'within_direction_seeds\': direction_seeds,\n                         \'outside_overlaps\': [{\'a\': a, \'b\': b, \'count\': len(set(masks[a]) & set(masks[b]))}\n                                              for a in range(3) for b in range(a+1,3)]}\n    return result\n\n\ndef arm_specs():\n    specs = [{\'name\': arm[\'name\'], \'kind\': \'baseline\', \'receiver\': arm[\'name\'], \'block\': None}\n             for arm in BASELINE_ARMS]\n    for block in BLOCKS:\n        for direction in (\'restore\', \'transfer\'):\n            receiver = PULSE_NAME[block] if direction == \'restore\' else \'sham\'\n            specs.append({\'name\': f\'b{block}_{direction}\', \'kind\': \'target\', \'block\': block,\n                          \'direction\': direction, \'receiver\': receiver})\n        for receiver, suffix in ((\'sham\', \'sham\'), (PULSE_NAME[block], \'pulse\')):\n            specs.append({\'name\': f\'b{block}_identity_{suffix}\', \'kind\': \'identity\', \'block\': block, \'receiver\': receiver})\n        for kind in (\'outside\', \'within\'):\n            for direction in (\'restore\', \'transfer\'):\n                for index in range(3):\n                    specs.append({\'name\': f\'b{block}_{kind}_{direction}_{index}\', \'kind\': kind,\n                        \'block\': block, \'direction\': direction, \'receiver\': PULSE_NAME[block] if direction == \'restore\' else \'sham\',\n                        \'control_index\': index})\n    if len(specs) != 35:\n        raise AssertionError(\'Expected exactly 35 prespecified prefill slots\')\n    return specs\n\n\ndef generated_names(request_id):\n    return set(BASELINE_NAMES + MAIN_GROUP_NAMES) if request_id in ORIGINAL_IDS else set(BASELINE_NAMES)\n\n\ndef plan_document(reference, stats, items, continuity, controls):\n    names = (\'attention_probe/group_restoration.py\', \'attention_probe/group_probe_items.json\')\n    return {\'schema\': SCHEMA, \'purpose\': \'same-prompt local H286 restoration/transfer; exploratory follow-up\',\n        \'helper_sha256\': base.sha_file(__file__),\n        \'new_source_sha256\': {name: base.sha_file(source_path(name)) for name in names},\n        \'unchanged_source_sha256\': continuity[\'source_module_sha256\'],\n        \'continuity_sha256\': CONTINUITY_SHA256, \'prior_archive_sha256\': continuity[\'source_archive_sha256\'],\n        \'prior_plan_sha256\': continuity[\'source_plan_sha256\'], \'model\': reference.model.to_dict(),\n        \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n        \'refw_sha256\': base.REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n        \'coordinates\': [list(pair) for pair in reference.coordinates], \'items\': items,\n        \'group_items_sha256\': GROUP_ITEMS_SHA256, \'seed\': base.SEED,\n        \'controls\': {str(key):value for key,value in controls.items()}, \'array_contract\': ARRAY_CONTRACT, \'prefill_arms\': arm_specs(), \'baseline_arms\': list(BASELINE_ARMS),\n        \'generation\': {\'max_new_tokens\': 128, \'original_names\': sorted(generated_names(\'r012\')),\n                       \'variant_names\': sorted(generated_names(\'r100\')), \'planned_count\': 46,\n                       \'hooks\': \'prefill only; fresh cache and original request seed per arm\'},\n        \'prefill_slots\': 350, \'native_uninstrumented_prefills\': 10, \'generation_replay_prefills\': 46,\n        \'planned_total_prefill_forwards\': 406, \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'sdpa/EFFICIENT_ATTENTION\',\n        \'transformers\': base.TRANSFORMERS, \'donors\': \'fresh current-session clean/pulse states of identical prompt only\',\n        \'normalization\': \'match raw delta-z norm; report projected Wdown norm/cosine and actual addition rounding\',\n        \'degeneracy\': \'zero target gives explicit zero no-op; nonzero target with degenerate donor/random direction invalid; no resampling\',\n        \'outside_direction\': \'actual same-prompt pulseZ_C-cleanZ_C, raw-norm scaled; not Gaussian\',\n        \'numeric_guards\': {\'zero_target\':\'exactly zero\',\'normalizer_min_norm\':1e-12,\n                           \'raw_norm_match_rtol\':1e-3,\'raw_norm_match_atol\':0.,\'cosine_denominator_min\':1e-24},\n        \'full_vocabulary_KL\': \'all vocabulary entries against each of three same-prompt baselines, both directions\',\n        \'semantic_metric\': \'correct-vs-foil FIRST TOKEN only, spelling/format-dependent; no semantic correctness verdict\',\n        \'queue_budget\': {\'wallclock_limit_seconds\': None, \'completion_guaranteed\': False},\n        \'claim\': \'No training; post-result follow-up, not confirmatory; output format and semantics distinct\'}\n\n\ndef prepare_request(tokenizer, index, item):\n    original = base.prepare_request(tokenizer, int(item[\'request_id\'][1:]), item)\n    keys = (\'request_id\',\'item_id\',\'pair_id\',\'category\',\'question\',\'answer\',\'formatted_prompt\',\n            \'formatted_prompt_sha256\',\'prompt_ids\',\'answer_ids\',\'thinking_enabled\')\n    request = {key: original[key] for key in keys}\n    position = len(request[\'prompt_ids\'])-1\n    request.update(group_item=dict(item), validation_schema=SCHEMA, pulse_positions=[position],\n                   prefill_capture_positions=[position], prefill_capture_scope=\'last prompt token only; no teacher-forced gold\',\n                   generation_capture_positions=list(range(max(0,position-3),position+1)))\n    if request[\'request_id\'] != item[\'request_id\']:\n        raise ValueError(\'Original/random-control request seed identity changed\')\n    return request\n\n\ndef tensor_sha(value):\n    return hashlib.sha256(value.detach().float().cpu().contiguous().numpy().tobytes()).hexdigest()\n\n\ndef receiver_arm(name):\n    return next(arm for arm in BASELINE_ARMS if arm[\'name\'] == name)\n\n\ndef make_patch(spec, donors, controls):\n    import torch\n    block = spec[\'block\']\n    setting = controls[block]\n    group = setting[\'group\']\n    clean = donors[\'sham\'][\'z\'][block][:, :, group]\n    pulse = donors[PULSE_NAME[block]][\'z\'][block][:, :, group]\n    delta_transfer = pulse-clean\n    raw_target = float(delta_transfer.double().norm())\n    sign = -1 if spec.get(\'direction\') == \'restore\' else 1\n    target_delta = sign*delta_transfer\n    record = {\'raw_target_delta_norm\': raw_target, \'target_delta_degenerate\': raw_target == 0,\n              \'same_prompt_donor\': True, \'donor_clean_sha256\': tensor_sha(donors[\'sham\'][\'z\'][block]),\n              \'donor_pulse_sha256\': tensor_sha(donors[PULSE_NAME[block]][\'z\'][block])}\n    if spec[\'kind\'] == \'target\':\n        donor = clean if spec[\'direction\'] == \'restore\' else pulse\n        return {\'channels\': group, \'donor_values\': donor}, target_delta, record\n    if spec[\'kind\'] == \'identity\':\n        donor = donors[spec[\'receiver\']][\'z\'][block][:, :, group]\n        return {\'channels\': group, \'donor_values\': donor}, torch.zeros_like(delta_transfer), record\n    index = spec[\'control_index\']\n    channels = setting[\'outside_masks\'][index] if spec[\'kind\'] == \'outside\' else group\n    if not math.isfinite(raw_target):\n        return None, target_delta, dict(record, invalid_reason=\'nonfinite target delta norm; no resampling\')\n    if raw_target == 0:\n        record[\'zero_target_noop\'] = True\n        return {\'channels\': channels, \'delta\': torch.zeros((1,1,len(channels)),dtype=torch.float32)}, target_delta, record\n    if spec[\'kind\'] == \'outside\':\n        direction = (donors[PULSE_NAME[block]][\'z\'][block][:,:,channels]-donors[\'sham\'][\'z\'][block][:,:,channels]).double().reshape(-1)\n        record[\'direction_source\'] = \'same-prompt actual pulseZ_C-cleanZ_C\'\n    else:\n        unit = setting[\'within_units\'][index]\n        if unit is None:\n            return None, target_delta, dict(record, invalid_reason=\'degenerate fixed random direction; no resampling\')\n        direction = torch.tensor(unit,dtype=torch.float64)\n        record[\'direction_source\'] = \'fixed within-G Gaussian unit vector\'\n    norm = float(direction.norm()); record[\'unscaled_direction_raw_norm\'] = norm\n    if not math.isfinite(norm) or norm <= 1e-12:\n        return None, target_delta, dict(record, invalid_reason=\'degenerate normalizing donor/random direction; no resampling\')\n    delta = (direction*(sign*raw_target/norm)).float()[None,None,:]\n    record[\'scale_requested\'] = sign*raw_target/norm\n    return {\'channels\': channels, \'delta\': delta}, target_delta, record\n\n\ndef jsonable(value):\n    if hasattr(value, \'detach\'):\n        return value.detach().cpu().tolist()\n    if isinstance(value, dict):\n        return {key: jsonable(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [jsonable(item) for item in value]\n    return value\n\n\ndef patch_summary(patch, model, spec, target_delta, controls, calibration):\n    if patch is None:\n        return None\n    record = dict(patch.record)\n    vector = record.pop(\'projected_delta_vector\').double().reshape(-1)\n    group = controls[spec[\'block\']][\'group\']\n    weight = model.model.layers[spec[\'block\']].mlp.down_proj.weight\n    desired = (weight[:, group] @ target_delta.to(weight.device).reshape(-1)).detach().double().cpu()\n    denominator = float(vector.norm()*desired.norm())\n    cosine = float(vector.dot(desired)/denominator) if denominator > 1e-24 else None\n    actual = float(record[\'raw_delta_norm\'].reshape(-1)[0])\n    target = calibration[\'raw_target_delta_norm\']\n    matched = (spec[\'kind\'] not in (\'outside\',\'within\') or (target == 0 and actual == 0)\n               or (target > 0 and actual > 0 and abs(actual-target) <= 1e-3*target))\n    return {**jsonable(record), **calibration, \'projected_cosine_to_target_delta\': cosine,\n            \'projected_cosine_valid\': cosine is not None, \'target_projected_delta_norm\':float(desired.norm()),\n            \'actual_direction_scale_norm\':actual/calibration[\'unscaled_direction_raw_norm\']\n                if calibration.get(\'unscaled_direction_raw_norm\',0)>0 else None, \'raw_norm_match_passed\': matched,\n            \'actual_to_target_raw_norm_ratio\': actual/target if target > 0 else None,\n            \'fired_count\': patch.fired_count, \'completed_count\': patch.completed_count,\n            \'removed_before_decode\': patch.removed_before_decode}\n\n\ndef run_prefill(model, request, coordinates, stats, arm, *, patch_args=None, efficient_gpu=True, weight_norms=None):\n    import torch\n\n    from attention_probe.group_restoration import GroupZPatch, ProbeCollector\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    ids = torch.tensor([request[\'prompt_ids\']], dtype=torch.long, device=next(model.parameters()).device)\n    positions = [len(request[\'prompt_ids\'])-1]\n    patch = None\n    with torch.inference_mode(), previous.attention_context(torch, efficient_gpu), contextlib.ExitStack() as stack:\n        # Patch PRE-hook precedes the readout PRE-hooks: all recorded z is post-patch.\n        if patch_args is not None:\n            patch = stack.enter_context(GroupZPatch(model, patch_args[\'block\'], positions,\n                patch_args[\'channels\'], **{k:v for k,v in patch_args.items() if k not in (\'block\', \'channels\')}))\n        probe = stack.enter_context(ProbeCollector(model, blocks=BLOCKS, positions=positions))\n        capture = stack.enter_context(CoordinateCapture(model, coordinates, positions, residual_layers=(), expected_count=len(coordinates)))\n        cett = stack.enter_context(base.cett_tap(model, coordinates, positions, weight_norms))\n        pulse = stack.enter_context(AttentionPulse(model, arm[\'block\'], positions, arm[\'beta\'], arm[\'mode\'],\n                                                   seed=base.SEED+int(request[\'request_id\'][1:])))\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=True)\n        del output  # Never reuse this prefill cache as a donor or a generation cache.\n        raw = capture.values.clone()\n        contribution = cett[\'cett\'].clone()\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        arrays = {\'raw_z\': raw.numpy(), \'cett\': contribution.numpy(), \'zcett\': ((contribution-mu)/sd).numpy(),\n                  \'down_output_norm_raw\': cett[\'denominator\'].numpy().copy()}\n    if (pulse.fired_count!=1 or capture.fired_count!=1 or probe.full_passes!=1 or probe.completed_count!=1\n        or (patch is not None and (patch.fired_count!=1 or patch.completed_count!=1))):\n        raise ValueError(\'Prefill pulse/readout/patch did not fire exactly once\')\n    logits = probe.first_logits.reshape(-1).detach().float().cpu()\n    if not bool(torch.isfinite(logits).all()):\n        raise ValueError(\'Nonfinite full-vocabulary prefill logits\')\n    return {\'logits\': logits, \'z\': probe.z_by_block, \'arrays\': arrays, \'patch\': patch,\n            \'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count}\n\n\ndef logit_metrics(logits, baselines, tokenizer, request):\n    import torch\n    logq = logits.double().log_softmax(-1)\n    top = torch.topk(logq, min(10, len(logq)))\n    fixed = int(baselines[\'sham\'][\'logits\'].argmax())\n    gold_ids = request[\'answer_ids\']\n    foil_ids = tokenizer(request[\'group_item\'][\'semantic_alternative\'], add_special_tokens=False)[\'input_ids\']\n    if not gold_ids or not foil_ids:\n        raise ValueError(\'Correct/foil strings must tokenize without silent replacement\')\n    gold, foil = int(gold_ids[0]), int(foil_ids[0])\n    kl = {}\n    for name, baseline in baselines.items():\n        logp = baseline[\'logits\'].double().log_softmax(-1)\n        kl[name] = {\'baseline_to_arm\': float((logp.exp()*(logp-logq)).sum()),\n                    \'arm_to_baseline\': float((logq.exp()*(logq-logp)).sum())}\n    return {\'logits_sha256\': tensor_sha(logits), \'full_vocab_kl\': kl,\n            \'fixed_clean_top1_token_id\': fixed, \'fixed_clean_top1_logprob\': float(logq[fixed]),\n            \'first_token_correct_id\': gold, \'first_token_foil_id\': foil,\n            \'correct_token_ids\': gold_ids, \'foil_token_ids\': foil_ids,\n            \'correct_text\': request[\'answer\'], \'foil_text\': request[\'group_item\'][\'semantic_alternative\'],\n            \'metric_dtype\': \'float64 full-vocabulary log_softmax/KL; generation logger remains original float32\',\n            \'first_token_correct_minus_foil_logprob\': float(logq[gold]-logq[foil]),\n            \'first_token_format_dependent\': True, \'first_token_contrast_informative\': gold != foil,\n            \'top10\': [{\'token_id\': int(index), \'text\': tokenizer.decode([int(index)], skip_special_tokens=False,\n                       clean_up_tokenization_spaces=False), \'logprob\': float(value)}\n                      for index,value in zip(top.indices.tolist(),top.values.tolist(),strict=True)]}\n\n\ndef run_generation(model, tokenizer, request, coordinates, stats, spec, patch_args, *, efficient_gpu=True, weight_norms=None):\n    import torch\n\n    from attention_probe.completion_128 import run_generation_arm\n    from attention_probe.group_restoration import GroupZPatch\n    patch = None\n    with previous.attention_context(torch,efficient_gpu), contextlib.ExitStack() as stack:\n        if patch_args is not None:\n            patch = stack.enter_context(GroupZPatch(model, patch_args[\'block\'], request[\'pulse_positions\'],\n                patch_args[\'channels\'], **{k:v for k,v in patch_args.items() if k not in (\'block\',\'channels\')}))\n        # The unchanged decoder enters CoordinateCapture AFTER the patch and removes\n        # its pulse/readout hooks before cached decode. Outer patch auto-removes too.\n        record, arrays = run_generation_arm(model, tokenizer, request, coordinates, stats,\n            receiver_arm(spec[\'receiver\']), efficient_gpu=efficient_gpu, weight_norms=weight_norms, max_new_tokens=128)\n    if patch is not None and not patch.removed_before_decode:\n        raise ValueError(\'Group patch was not removed before cached decoding\')\n    return record, arrays\n\n\ndef continuity_checks(request, rows, continuity, arrays):\n    if request[\'request_id\'] not in ORIGINAL_IDS:\n        return {\'scope\': \'new prespecified variants; no prior output reference\'}\n    old = continuity[\'requests\'][request[\'request_id\']]\n    completion.check_original_request(request, old)\n    checks = {}\n    for name in BASELINE_NAMES:\n        prior = old[\'arms\'][name]\n        generation = rows[name][\'generation\'][\'record\']\n        old_gen = prior[\'generation\']\n        protocol = all(generation[key] == old_gen[key] for key in\n                       (\'seed\',\'prompt_length\',\'eos_token_ids\',\'eos_source\',\'stop_reason\',\'selected_token_ids\'))\n        import numpy as np\n        prefix = \'generation_\'+name+\'_\'\n        saved = {\'gen_\'+key[len(prefix):]:value for key,value in arrays.items() if key.startswith(prefix)}\n        hashes = {key:hashlib.sha256(np.asarray(value).tobytes(order=\'C\')).hexdigest() for key,value in saved.items()}\n        checks[name] = {\'tokens_and_protocol_equal\': protocol,\n            \'prefill_arrays_exact\': hashes == prior[\'gen_array_sha256\'],\n            \'prefill_shapes_exact\': {key:list(value.shape) for key,value in saved.items()} == prior[\'gen_array_shapes\'],\n            \'prefill_array_sha256\': hashes,\n            \'first_logits_exact\': generation[\'first_step_logits_sha256\'] == old_gen[\'first_step_logits_sha256\'],\n            \'steps_exact\': hashlib.sha256(base.canonical(generation[\'steps\'])).hexdigest() == prior[\'steps_sha256\']}\n    return checks\n\n\ndef compute_input(engine, reference, stats, request, controls, continuity, *, efficient_gpu=True):\n    import numpy as np\n    import torch\n    model, tokenizer = engine[\'model\'], engine[\'tokenizer\']\n    specs = arm_specs()\n    donors, records, arrays, patches = {}, {}, {}, {}\n    initial_hooks = previous.hook_count(model)\n    if initial_hooks != 0: raise ValueError(\'Native clean control requires genuinely uninstrumented model\')\n    ids = torch.tensor([request[\'prompt_ids\']],dtype=torch.long,device=next(model.parameters()).device)\n    with torch.inference_mode(), previous.attention_context(torch,efficient_gpu):\n        native_output = model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=True)\n        native_logits = native_output.logits[0,-1].detach().float().cpu().clone()\n        del native_output\n    native_hash = tensor_sha(native_logits)\n    for spec in specs[:3]:\n        name = spec[\'name\']\n        result = run_prefill(model, request, reference.coordinates, stats, receiver_arm(name),\n                             efficient_gpu=efficient_gpu, weight_norms=engine.get(\'weight_norms\'))\n        donors[name] = result\n        records[name] = {\'name\': name, \'spec\': spec, \'state\': \'complete\', \'prefill_executed\': True, \'patch\': None}\n        arrays[f\'baseline_{name}_logits\'] = result[\'logits\'].numpy()\n        for block in BLOCKS:\n            arrays[f\'baseline_{name}_z_block_{block}\'] = result[\'z\'][block].numpy()\n        for key,value in result[\'arrays\'].items(): arrays[f\'prefill_{name}_{key}\'] = value\n    if native_hash != tensor_sha(donors[\'sham\'][\'logits\']): raise ValueError(\'Uninstrumented clean prefill differs from passive sham\')\n    print(f\'Stage {request["request_id"]}: frische Donors; feste Kontrollkalibrierung.\',flush=True)\n    # All masks/directions are already fixed in the plan. All 32 raw-norm\n    # calibrations are constructed before any control outcome or free generation.\n    calibration = {}\n    for spec in specs[3:]:\n        args, target, record = make_patch(spec, donors, controls)\n        if args is not None: args = dict(args, block=spec[\'block\'])\n        calibration[spec[\'name\']] = (args, target, record)\n    for name in BASELINE_NAMES:\n        records[name].update(logit_metrics(donors[name][\'logits\'], donors, tokenizer, request))\n    order = list(range(3,35)); random.Random(base.SEED+int(request[\'request_id\'][1:])).shuffle(order)\n    for index in order:\n        spec=specs[index];name=spec[\'name\'];args,target,cal=calibration[name]\n        patches[name]=args\n        if args is None:\n            records[name]={\'name\':name,\'spec\':spec,\'state\':\'invalid\',\'prefill_executed\':False,\'reason\':cal[\'invalid_reason\'],\'calibration\':cal}\n            continue\n        result=run_prefill(model,request,reference.coordinates,stats,receiver_arm(spec[\'receiver\']),\n            patch_args=args,efficient_gpu=efficient_gpu,weight_norms=engine.get(\'weight_norms\'))\n        summary=patch_summary(result[\'patch\'],model,spec,target,controls,cal)\n        records[name]={\'name\':name,\'spec\':spec,\'state\':\'complete\' if summary[\'raw_norm_match_passed\'] else \'invalid\',\n                       \'prefill_executed\':True,\'patch\':summary,\n                       **logit_metrics(result[\'logits\'],donors,tokenizer,request)}\n        for key,value in result[\'arrays\'].items():arrays[f\'prefill_{name}_{key}\']=value\n        if spec[\'kind\']==\'identity\' and records[name][\'logits_sha256\']!=records[spec[\'receiver\']][\'logits_sha256\']:\n            raise ValueError(\'Identity group patch changed logits\')\n    print(f\'Stage {request["request_id"]}: alle 35 Prefill-Slots abgeschlossen; vorgeplante Generierungen.\',flush=True)\n    generation_names=generated_names(request[\'request_id\'])\n    for spec in specs:\n        name=spec[\'name\'];row=records[name]\n        if name not in generation_names:\n            row[\'generation\']={\'state\':\'not_run\',\'reason\':\'not selected by fixed generation plan\'}\n            continue\n        if row[\'state\']!=\'complete\':raise ValueError(\'A main generation arm is invalid; no adaptive replacement\')\n        gen,gen_arrays=run_generation(model,tokenizer,request,reference.coordinates,stats,spec,patches.get(name),\n            efficient_gpu=efficient_gpu,weight_norms=engine.get(\'weight_norms\'))\n        if gen[\'first_step_logits_sha256\']!=row[\'logits_sha256\']:\n            raise ValueError(\'Generation first logits differ from corresponding measured prefill arm: \'+name)\n        row[\'generation\']={\'state\':\'complete\',\'record\':gen}\n        for key,value in gen_arrays.items():arrays[f\'generation_{name}_{key}\']=value\n    check=continuity_checks(request,records,continuity,arrays)\n    if any(row.get(\'tokens_and_protocol_equal\') is False for row in check.values() if isinstance(row,dict)):\n        raise ValueError(\'Fresh original baseline token/protocol differs from the bound 128-token run\')\n    for value in arrays.values():\n        if np.asarray(value).dtype!=np.float32 or not np.isfinite(value).all():raise ValueError(\'Nonfinite/non-FP32 group readout\')\n    if previous.hook_count(model) != initial_hooks: raise ValueError(\'Group probe left model hooks installed\')\n    return {\'technical_checks\': {\'state\':\'passed\',\'checks\':{\'native_clean_logits\':True,\'identity_logits\':True,\n            \'prefill_counts\':True,\'hook_cleanup\':True,\'generation_prefill_logits\':True,\'fresh_same_prompt_donors\':True},\n            \'native_clean_first_logits_sha256\':native_hash},\'request\':{k:v for k,v in request.items() if k!=\'ids\'},\'arms\':[records[s[\'name\']] for s in specs],\n            \'prefill_execution_order\':list(BASELINE_NAMES)+[specs[i][\'name\'] for i in order],\n            \'continuity\':check,\'planned_prefill_slots\':35,\'executed_prefills\':sum(r[\'prefill_executed\'] for r in records.values()),\n            \'generation_count\':len(generation_names),\'array_contract\':{\'vocab_size\':int(model.config.vocab_size),\n            \'hidden_size\':int(model.config.hidden_size),\'mlp_width\':int(model.config.intermediate_size),\n            \'n_coordinates\':len(reference.coordinates)}},arrays\n\n\ndef frozen_items():\n    path=source_path(\'attention_probe/group_probe_items.json\')\n    if base.sha_file(path)!=GROUP_ITEMS_SHA256:raise ValueError(\'Frozen group item source changed\')\n    return {item[\'request_id\']:item for item in json.loads(path.read_text())}\n\n\ndef validate_record(record, plan_hash, request_id):\n    if (record.get(\'schema\')!=SCHEMA or record.get(\'state\')!=\'complete\' or record.get(\'plan_sha256\')!=plan_hash\n        or record.get(\'request_id\')!=request_id or record.get(\'planned_prefill_slots\')!=35\n        or record.get(\'generation_count\')!=len(generated_names(request_id))\n        or record.get(\'array_contract\')!=ARRAY_CONTRACT):\n        raise ValueError(\'Incomplete or differently bound group input checkpoint\')\n    request=record.get(\'request\',{});item=frozen_items()[request_id]\n    if (request.get(\'request_id\')!=request_id or request.get(\'group_item\')!=item\n        or request.get(\'item_id\')!=item[\'id\'] or request.get(\'question\')!=item[\'prompt\']\n        or request.get(\'answer\')!=item[\'target\'] or request.get(\'validation_schema\')!=SCHEMA):\n        raise ValueError(\'Checkpoint request identity/frozen item changed\')\n    ids=request.get(\'prompt_ids\')\n    if not isinstance(ids,list) or not ids or any(type(value) is not int or not 0<=value<ARRAY_CONTRACT[\'vocab_size\'] for value in ids):\n        raise ValueError(\'Missing or invalid original prompt token IDs\')\n    positions=[len(ids)-1]\n    if (request.get(\'pulse_positions\')!=positions or request.get(\'prefill_capture_positions\')!=positions\n        or request.get(\'generation_capture_positions\')!=list(range(max(0,len(ids)-4),len(ids)))\n        or hashlib.sha256(request[\'formatted_prompt\'].encode()).hexdigest()!=request.get(\'formatted_prompt_sha256\')):\n        raise ValueError(\'Prompt-only prefill/generation positions or template hash changed\')\n    proof=record.get(\'technical_checks\',{})\n    if (proof.get(\'state\')!=\'passed\' or proof.get(\'plan_sha256\')!=plan_hash\n        or any(proof.get(\'checks\',{}).get(name) is not True for name in PROOF_CHECKS)):\n        raise ValueError(\'Missing plan-bound group technical checks\')\n    rows=record.get(\'arms\',[])\n    if [row.get(\'spec\') for row in rows]!=arm_specs():raise ValueError(\'Group arm plan changed\')\n    for row,spec in zip(rows,arm_specs(),strict=True):\n        if row.get(\'name\')!=spec[\'name\'] or type(row.get(\'prefill_executed\')) is not bool:\n            raise ValueError(\'Group arm identity/prefill execution marker changed\')\n        expected=\'complete\' if row[\'name\'] in generated_names(request_id) else \'not_run\'\n        if row.get(\'generation\',{}).get(\'state\')!=expected:raise ValueError(\'Group generation/not_run contract changed\')\n        if row.get(\'state\') not in (\'complete\',\'invalid\') or (row[\'state\']==\'complete\' and not row[\'prefill_executed\']):\n            raise ValueError(\'Unfinished prefill slot\')\n        if row[\'state\']==\'invalid\' and spec[\'kind\'] not in (\'outside\',\'within\'):\n            raise ValueError(\'A fixed main/identity arm cannot be replaced by an invalid control\')\n        if row[\'state\']==\'complete\' and spec[\'kind\'] in (\'outside\',\'within\') and row.get(\'patch\',{}).get(\'raw_norm_match_passed\') is not True:\n            raise ValueError(\'An unmatched control cannot enter the valid comparison\')\n        if row[\'prefill_executed\']:\n            if (not isinstance(row.get(\'logits_sha256\'),str) or len(row[\'logits_sha256\'])!=64\n                or set(row.get(\'full_vocab_kl\',{}))!=set(BASELINE_NAMES)):\n                raise ValueError(\'Missing full-vocabulary KL endpoints/hash\')\n            for value in row[\'full_vocab_kl\'].values():\n                if set(value)!= {\'baseline_to_arm\',\'arm_to_baseline\'} or any(not math.isfinite(x) for x in value.values()):\n                    raise ValueError(\'Nonfinite/incomplete full-vocabulary KL\')\n        if expected==\'complete\':\n            generation=row[\'generation\'][\'record\']\n            completion.FUNCTIONS[\'validate_generation\'](generation,request,receiver_arm(spec[\'receiver\']))\n            if generation.get(\'seed\')!=base.SEED+int(request_id[1:]):\n                raise ValueError(\'Original per-request generation seed changed\')\n            if generation.get(\'first_step_logits_sha256\')!=row[\'logits_sha256\']:\n                raise ValueError(\'Generation/prefill logit identity changed\')\n    if record.get(\'executed_prefills\')!=sum(row[\'prefill_executed\'] for row in rows):\n        raise ValueError(\'Executed prefill count changed\')\n\n\ndef validate_arrays(record, arrays):\n    import numpy as np\n    contract=ARRAY_CONTRACT;expected={};request=record[\'request\']\n    for name in BASELINE_NAMES:\n        expected[f\'baseline_{name}_logits\']=(contract[\'vocab_size\'],)\n        for block in BLOCKS:expected[f\'baseline_{name}_z_block_{block}\']=(1,1,contract[\'mlp_width\'])\n    for row in record[\'arms\']:\n        name=row[\'name\']\n        if row[\'prefill_executed\']:\n            for key in (\'raw_z\',\'cett\',\'zcett\',\'down_output_norm_raw\'):\n                expected[f\'prefill_{name}_{key}\']=(1,1,contract[\'n_coordinates\'])\n        if row[\'generation\'][\'state\']==\'complete\':\n            length=min(4,len(request[\'prompt_ids\']))\n            for key in (\'raw_z\',\'cett\',\'zcett\',\'down_output_norm_raw\'):\n                expected[f\'generation_{name}_{key}\']=(1,length,contract[\'n_coordinates\'])\n            for block in (3,7,14,27):expected[f\'generation_{name}_residual_layer_{block}\']=(1,length,contract[\'hidden_size\'])\n    keys=set(arrays.files) if hasattr(arrays,\'files\') else set(arrays)\n    if keys!=set(expected):raise ValueError(\'Missing or unexpected group NPZ measurement keys\')\n    for key,shape in expected.items():\n        value=np.asarray(arrays[key])\n        if value.shape!=shape or value.dtype!=np.float32 or not np.isfinite(value).all():\n            raise ValueError(\'Group array shape/dtype/finiteness mismatch: \'+key)\n    rows={row[\'name\']:row for row in record[\'arms\']}\n    for name in BASELINE_NAMES:\n        value=np.asarray(arrays[f\'baseline_{name}_logits\'])\n        if hashlib.sha256(value.tobytes(order=\'C\')).hexdigest()!=rows[name][\'logits_sha256\']:\n            raise ValueError(\'Saved baseline full-vocabulary endpoint hash changed\')\n\n\ndef verified_input(output, request_id, plan_hash, continuity=None):\n    import numpy as np\n    directory=Path(output)/\'requests\'/request_id\n    if not directory.exists():return None\n    record=json.loads((directory/\'result.json\').read_text());validate_record(record,plan_hash,request_id)\n    if base.sha_file(directory/\'arrays.npz\')!=record[\'npz_sha256\']:raise ValueError(\'Group checkpoint NPZ SHA mismatch\')\n    with np.load(directory/\'arrays.npz\',allow_pickle=False) as saved:\n        validate_arrays(record,saved)\n        if continuity is not None:\n            actual=continuity_checks(record[\'request\'],{row[\'name\']:row for row in record[\'arms\']},continuity,saved)\n            if actual!=record[\'continuity\']:raise ValueError(\'Stored original continuity evidence changed\')\n    return record\n\n\ndef write_input(output, request_id, result, arrays, plan_hash, hardware, proof):\n    import numpy as np\n    parent=Path(output)/\'requests\';parent.mkdir(parents=True,exist_ok=True)\n    destination=parent/request_id\n    if destination.exists():raise ValueError(\'Refusing to overwrite completed group input\')\n    temporary=Path(tempfile.mkdtemp(prefix=\'.\'+request_id+\'_\',dir=parent))\n    try:\n        np.savez_compressed(temporary/\'arrays.npz\',**arrays)\n        record=dict(result,schema=SCHEMA,state=\'complete\',request_id=request_id,plan_sha256=plan_hash,\n                    npz_sha256=base.sha_file(temporary/\'arrays.npz\'),hardware=hardware,technical_checks=proof)\n        validate_record(record,plan_hash,request_id)\n        validate_arrays(record,arrays)\n        base.atomic_json(temporary/\'result.json\',record)\n        os.rename(temporary,destination)\n    finally:\n        if temporary.exists():shutil.rmtree(temporary)\n    return record\n\n\ndef restore_resume(path,output,plan_hash,continuity=None):\n    if not path:return\n    with zipfile.ZipFile(path) as packed:\n        plan=json.loads(packed.read(\'run_plan.json\'))\n        if plan.get(\'schema\')!=SCHEMA or hashlib.sha256(base.canonical(plan)).hexdigest()!=plan_hash:\n            raise ValueError(\'Resume has another group/source/reference plan\')\n        for rid in REQUEST_IDS:\n            prefix=\'requests/\'+rid+\'/\'\n            names=[prefix+\'result.json\',prefix+\'arrays.npz\'];present=[n in packed.namelist() for n in names]\n            if not any(present):continue\n            if not all(present):raise ValueError(\'Incomplete group input in resume\')\n            parent=Path(output)/\'requests\';parent.mkdir(parents=True,exist_ok=True);dest=parent/rid\n            if dest.exists():\n                verified_input(output,rid,plan_hash,continuity)\n                if any((dest/Path(n).name).read_bytes()!=packed.read(n) for n in names):raise ValueError(\'Resume conflicts; no overwrite\')\n                continue\n            temporary=Path(tempfile.mkdtemp(prefix=\'.\'+rid+\'_\',dir=parent))\n            try:\n                for name in names:\n                    if packed.getinfo(name).file_size>64*(1<<20):raise ValueError(\'Oversized group checkpoint\')\n                    (temporary/Path(name).name).write_bytes(packed.read(name))\n                rec=json.loads((temporary/\'result.json\').read_text());validate_record(rec,plan_hash,rid)\n                if base.sha_file(temporary/\'arrays.npz\')!=rec[\'npz_sha256\']:raise ValueError(\'Resume group arrays SHA mismatch\')\n                import numpy as np\n                with np.load(temporary/\'arrays.npz\',allow_pickle=False) as arrays:validate_arrays(rec,arrays)\n                os.rename(temporary,dest);verified_input(output,rid,plan_hash,continuity)\n            finally:\n                if temporary.exists():shutil.rmtree(temporary)\n\n\ndef archive(output):\n    target=Path(output)/\'h286_group_restore_results.zip\';temporary=target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temporary,\'w\',zipfile.ZIP_DEFLATED) as packed:\n        for path in sorted(Path(output).rglob(\'*\')):\n            if path.is_file() and path.suffix not in (\'.zip\',\'.tmp\') and not any(p.startswith(\'.\') for p in path.relative_to(output).parts):\n                packed.write(path,path.relative_to(output))\n    os.replace(temporary,target)\n\n\ndef run(hn,policy,refw,items_path,continuity_path,out_dir,cache_dir=None,resume_zip=None):\n    output=Path(out_dir);output.mkdir(parents=True,exist_ok=True);started=time.monotonic()\n    status={\'state\':\'running\',\'stage\':\'reference\',\'scope\':SCHEMA,\'completed_inputs\':0,\'confirmatory_eligible\':False}\n    metrics={\'weights_loaded\':False,\'pretrained_model_run\':False,\'wrapper_attempt_id\':os.environ.get(\'H286_WRAPPER_ATTEMPT\'),\n             \'no_training\':True,\'no_confirmatory_claim\':True}\n    def save():base.atomic_json(output/\'status.json\',status);base.atomic_json(output/\'metrics.json\',metrics)\n    save()\n    try:\n        reference,stats,items,continuity=load_inputs(hn,policy,refw,items_path,continuity_path)\n        controls=make_controls(reference.coordinates,reference.model.mlp_width)\n        plan=plan_document(reference,stats,items,continuity,controls);digest=hashlib.sha256(base.canonical(plan)).hexdigest()\n        path=output/\'run_plan.json\'\n        if path.exists() and path.read_bytes()!=base.canonical(plan)+b\'\\n\':raise ValueError(\'Different group plan; do not mix outputs\')\n        if not path.exists():base.atomic_json(path,plan)\n        restore_resume(resume_zip,output,digest,continuity)\n        done={rid:verified_input(output,rid,digest,continuity) for rid in REQUEST_IDS}\n        if not all(done.values()):\n            status[\'stage\']=\'hardware_and_original_model_load\';save()\n            def loaded(hardware):metrics.update(weights_loaded=True,hardware=hardware);save()\n            engine=previous.initialize_engine(reference,Path(cache_dir or Path.home()/\'.cache/h286_pilot\'),on_loaded=loaded)\n            import torch\n            model=engine[\'model\']\n            actual={\'vocab_size\':int(model.config.vocab_size),\'hidden_size\':int(model.config.hidden_size),\n                    \'mlp_width\':int(model.config.intermediate_size),\'n_coordinates\':len(reference.coordinates)}\n            if actual!=ARRAY_CONTRACT:raise ValueError(\'Original fixed model/array architecture changed\')\n            frozen=base.parameter_evidence(engine[\'model\']);measured=[]\n            for index,item in enumerate(items):\n                rid=item[\'request_id\']\n                if done[rid] is not None:continue\n                request=prepare_request(engine[\'tokenizer\'],index,item)\n                status.update(stage=\'prefills_and_prespecified_generations\',current_request=rid);save()\n                print(f\'Request {index+1}/10 {rid}: 35 Prefill-Slots, feste Gruppen vor Generierung.\',flush=True)\n                cpu_rng=torch.random.get_rng_state().clone();cuda_rng=torch.cuda.get_rng_state().clone();t0=time.monotonic()\n                result,arrays=compute_input(engine,reference,stats,request,controls,continuity)\n                if base.parameter_evidence(engine[\'model\'])!=frozen or not torch.equal(cpu_rng,torch.random.get_rng_state()) or not torch.equal(cuda_rng,torch.cuda.get_rng_state()):\n                    raise ValueError(\'Frozen parameters/global RNG changed during group input\')\n                metrics[\'pretrained_model_run\']=True\n                proof=dict(result[\'technical_checks\'],plan_sha256=digest)\n                proof[\'checks\'].update(frozen_parameters=True,global_rng=True)\n                done[rid]=write_input(output,rid,result,arrays,digest,engine[\'hardware\'],proof)\n                measured.append(time.monotonic()-t0);remaining=sum(v is None for v in done.values())\n                status.update(completed_inputs=10-remaining,estimated_remaining_seconds=sum(measured)/len(measured)*remaining);save()\n                print(f\'Gespeichert {10-remaining}/10; Restschätzung {status["estimated_remaining_seconds"]/60:.1f} min.\',flush=True)\n        metrics[\'generation_count\']=sum(row[\'generation_count\'] for row in done.values())\n        metrics[\'prefill_slots\']=sum(row[\'planned_prefill_slots\'] for row in done.values())\n        metrics[\'native_uninstrumented_prefills\']=10\n        metrics[\'generation_replay_prefills\']=46\n        metrics[\'executed_measurement_prefills\']=sum(row[\'executed_prefills\'] for row in done.values())\n        metrics[\'total_prefill_forwards\']=metrics[\'executed_measurement_prefills\']+10+46\n        if metrics[\'generation_count\']!=46 or metrics[\'prefill_slots\']!=350:raise ValueError(\'Fixed group-run cardinality changed\')\n        status.update(state=\'complete\',stage=\'complete\',completed_inputs=10)\n    except BaseException as error:  # noqa: BLE001 -- keep diagnostic ZIP on interruption.\n        status.update(state=\'failed\',error_type=type(error).__name__,error=str(error));(output/\'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \'+str(error),flush=True)\n    finally:\n        metrics[\'elapsed_seconds\']=time.monotonic()-started;save();archive(output)\n    return status\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in (\'hn-set\',\'policy\',\'refw\',\'items\',\'continuity\',\'out-dir\'):parser.add_argument(\'--\'+name,required=True)\n    parser.add_argument(\'--cache-dir\');parser.add_argument(\'--resume-zip\');args=parser.parse_args()\n    result=run(args.hn_set,args.policy,args.refw,args.items,args.continuity,args.out_dir,args.cache_dir,args.resume_zip)\n    print(json.dumps(result),flush=True);raise SystemExit(0 if result[\'state\']==\'complete\' else 1)\n'}
MODULE_SHA256 = {'attention_probe/__init__.py': 'de626db34130b6d517d0dc776df8eff3b794bec5d7c9ba70157abe016f422705', 'attention_probe/reference.py': 'e89bd465d0e962e840ba589280ae89b84dd74f6fda1a459f4c76826ef4bd51e3', 'attention_probe/fixed_refw.py': 'b5cb362889f473f8b902454b4e22f4a615a54ed0a2bac0d7b0cc4c80e3c778e3', 'attention_probe/interventions.py': '1c7afb046108e2d7a214e1849198574128acf2c3e300888cdb6cc47e0e086561', 'attention_probe/pilot_items.json': '12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3', 'qwen3_h286_runtime.py': 'e30b9da80bd9370eb5de04245ad43b8c0ac0e292611ba32d1a8721b28e1e4312', 'qwen3_h286_validation_runtime.py': '7ab81c384801adf466f5ef22c20280b66c2df86b0a684fda0c5d1f9044632ba6', 'attention_probe/answer_validation.py': '9bac92b5e286388d2f1e8cbf71b9fc181840de8b60f0988360eb9e5692a1ba3c', 'attention_probe/completion_128.py': '15123bd828d47228583819b12fa80deaa51b92a63f8ca5a38a5bd72665099e75', 'qwen3_h286_completion_128_runtime.py': '251dc72efae8e9ac21b525a12f6b157f66a9475b17fab4615062dca64af45147', 'attention_probe/group_restoration.py': 'b95f548ee1195a754abf694a4fa08213b331e85aadea3ee7fe0888f26823237c', 'attention_probe/group_probe_items.json': '0bf76a995436092728cacc46e1be78c92b8e5416b66402324687b2a796b8ec88', 'qwen3_h286_group_runtime.py': 'c928317b41f8f4dc6ad0334779ecaf0d4ca4103442855f0642f8e98691591086'}


In [ ]:
import base64
from collections import deque
import datetime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import traceback
import zipfile
import uuid

wrapper_attempt_id = uuid.uuid4().hex
base = Path('/content') if Path('/content').is_dir() else Path.cwd()
binding_fingerprint = hashlib.sha256(json.dumps({'modules': MODULE_SHA256, 'references': REFERENCE_SHA256},
                                                sort_keys=True).encode()).hexdigest()[:12]
work = (Path(RUN_DIRECTORY).expanduser() if RUN_DIRECTORY
        else base / ('h286_group_restore_' + binding_fingerprint))
output = work / 'output'
output.mkdir(parents=True, exist_ok=True)
failure = None
helper_started = False
helper_status = None
return_code = None
status = {'state': 'running', 'stage': 'gpu_preflight', 'purpose': 'original_qwen3_h286_group_restoration',
          'pretrained_weights_loaded': False, 'inference': 'exploratory (post hoc)'}

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + chr(10))

def run_command(command, logname, env=None):
    process = None
    tail = deque(maxlen=10)
    shown = 0
    try:
        with (output / logname).open('a', encoding='utf-8') as log:
            process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                log.write(line); log.flush()
                short = line.strip().replace(chr(13), ' ')[:500]
                tail.append(short)
                if logname == 'runtime.log' and shown < 120 and short.startswith(
                        ('Request ', 'Gespeichert ', 'Stage ', 'Status:', 'Status ', 'CHECK ', 'HARD FAILURE:')):
                    print(short, flush=True); shown += 1
            code = process.wait()
        if code:
            print('Kindprozess fehlgeschlagen; vollständiges Log im ZIP. Letzte Zeilen:', flush=True)
            for short in tail:
                print(short, flush=True)
            raise subprocess.CalledProcessError(code, command)
        return code
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        raise

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Keine CUDA-GPU. A100, L4 oder B200 auswählen; kein Warten.')
    capability = tuple(torch.cuda.get_device_capability(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    if capability[0] < 8 or free_bytes < MIN_FREE_VRAM_GIB * 1024**3:
        raise RuntimeError('Ampere-oder-neuere CUDA-GPU mit mindestens 12 GiB freiem Speicher erforderlich.')
    torch_before = importlib.metadata.version('torch')
    environment = {'python': sys.version, 'torch': torch_before, 'torch_cuda': torch.version.cuda,
                   'gpu': torch.cuda.get_device_name(0), 'capability': list(capability),
                   'free_vram_gib': free_bytes / 1024**3, 'total_vram_gib': total_bytes / 1024**3,
                   'transformers_required': TRANSFORMERS_VERSION, 'accelerate_required': ACCELERATE_VERSION,
                   'execution_dtype': 'float32', 'tf32': False}
    write_json(output / 'environment_before.json', environment)
    print('GPU:', environment['gpu'], '| frei:', round(environment['free_vram_gib'], 2), 'GiB')
    print('Torch/CUDA bleibt erhalten:', torch_before, torch.version.cuda)
    status['stage'] = 'reference_validation'
    inputs = work / 'inputs'; inputs.mkdir(exist_ok=True)
    for name, expected in REFERENCE_SHA256.items():
        if name in REFERENCE_FILES_B64:
            raw = base64.b64decode(REFERENCE_FILES_B64[name], validate=True)
        elif REFERENCE_DIRECTORY:
            raw = (Path(REFERENCE_DIRECTORY).expanduser() / name).read_bytes()
        else:
            raise RuntimeError('Öffentliches Template ohne private Referenz. Die bereitgestellte private Nutzdatei verwenden.')
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Originalreferenz-Prüfsumme weicht ab: ' + name)
        target = inputs / name
        if target.exists() and target.read_bytes() != raw:
            raise ValueError('Vorhandene Referenz gehört zu einer anderen Auswahl; Ergebnisse nicht mischen.')
        target.write_bytes(raw)
    binding = {'sha256': MODULE_SHA256, 'reference_sha256': REFERENCE_SHA256,
               'model': 'Qwen/Qwen3-1.7B', 'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'}
    binding_path = output / 'embedded_source.json'
    if binding_path.exists() and json.loads(binding_path.read_text()) != binding:
        raise ValueError('Vorhandener Lauf gehört zu anderem Quellcode oder Referenzen; Ergebnisse nicht mischen.')
    code = work / 'code'; code.mkdir(exist_ok=True)
    for name, source in MODULE_SOURCES.items():
        if hashlib.sha256(source.encode()).hexdigest() != MODULE_SHA256[name]:
            raise ValueError('Eingebetteter Quellcode weicht ab: ' + name)
        path = code / name; path.parent.mkdir(parents=True, exist_ok=True); path.write_text(source)
    write_json(binding_path, binding)
    status['stage'] = 'dependencies'
    constraint = work / 'preserve_torch.txt'
    constraint.write_text('torch==' + torch_before + chr(10))
    print('Exakte Transformers-/Accelerate-Version installieren; GPU-Torch bleibt festgelegt.', flush=True)
    child_env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1',
                     TOKENIZERS_PARALLELISM='false')
    run_command([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--upgrade-strategy', 'only-if-needed',
                 '--constraint', str(constraint), 'transformers==' + TRANSFORMERS_VERSION,
                 'accelerate==' + ACCELERATE_VERSION], 'install.log', child_env)
    installed = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate')}
    if installed != {'torch': torch_before, 'transformers': TRANSFORMERS_VERSION, 'accelerate': ACCELERATE_VERSION}:
        raise RuntimeError('Paketversionen weichen ab; keine Modellgewichte angefordert.')
    write_json(output / 'environment_installed.json', installed)
    status['stage'] = 'runtime'
    helper_started = True
    status['pretrained_weights_loaded'] = False
    env = dict(child_env, PYTHONPATH=str(code), H286_WRAPPER_ATTEMPT=wrapper_attempt_id)
    command = [sys.executable, str(code / 'qwen3_h286_group_runtime.py'), '--hn-set', str(inputs / 'hn_set.json'),
               '--policy', str(inputs / 'htq_policy.json'), '--refw', str(inputs / 'refw_h286.json'),
               '--items', str(code / 'attention_probe/group_probe_items.json'),
               '--out-dir', str(output), '--cache-dir', str(base / 'h286_model_cache')]
    command += ['--continuity', str(inputs / 'group_continuity.json')]
    if RESUME_ZIP:
        command += ['--resume-zip', str(Path(RESUME_ZIP).expanduser())]
    print('Frischer Python-Prozess: Original-FP32-Modell, feste H286/REF-W, zehn Eingaben, begrenzte Gruppenrücksetzung und Antwortläufe.', flush=True)
    return_code = run_command(command, 'runtime.log', env)
    helper_status = json.loads((output / 'status.json').read_text())
    if helper_status.get('state') != 'complete':
        raise RuntimeError('Runtime hat keinen vollständigen Abschluss gemeldet.')
    status.update(helper_status, helper_return_code=return_code, wrapper_verified=True)
except BaseException as error:
    failure = error
    status.update(state='failed', error_type=type(error).__name__, error=str(error), wrapper_verified=False)
    (output / 'wrapper_diagnostic.txt').write_text(traceback.format_exc())
finally:
    # Keep the child's original status for diagnosis, then publish one authoritative final status.
    child_path = output / 'status.json'
    if helper_started and child_path.exists():
        try:
            helper_status = json.loads(child_path.read_text())
            write_json(output / 'helper_status.json', helper_status)
            if failure is not None:
                status['helper_state'] = helper_status.get('state')
                status['pretrained_weights_loaded'] = helper_status.get('pretrained_weights_loaded')
        except Exception:
            status['helper_state'] = 'unreadable'
    # Only this attempt's metrics can establish whether its model weights loaded.
    # Stable work folders may contain metrics from an earlier successful attempt.
    if helper_started:
        status['pretrained_weights_loaded'] = None
        status['weights_loaded_evidence'] = 'missing_unreadable_or_stale_child_metrics'
        try:
            child_metrics = json.loads((output / 'metrics.json').read_text())
            loaded = child_metrics.get('weights_loaded')
            if (child_metrics.get('wrapper_attempt_id') == wrapper_attempt_id
                    and type(loaded) is bool):
                status['pretrained_weights_loaded'] = loaded
                status['weights_loaded_evidence'] = 'fresh_child_metrics'
        except (OSError, ValueError, TypeError, AttributeError):
            pass
    status['wrapper_attempt_id'] = wrapper_attempt_id
    status['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(output / 'status.json', status)
    members = {str(path.relative_to(output)): hashlib.sha256(path.read_bytes()).hexdigest()
               for path in sorted(output.rglob('*')) if path.is_file()
               and path.name not in ('WRAPPER_MANIFEST.json',) and path.suffix not in ('.zip', '.tmp')
               and not any(part.startswith('.') for part in path.relative_to(output).parts)}
    write_json(output / 'WRAPPER_MANIFEST.json', members)
    target = base / ('h286_group_restore_diagnostic.zip' if failure is not None else 'h286_group_restore_results.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if (path.is_file() and path.suffix not in ('.zip', '.tmp')
                    and not any(part.startswith('.') for part in path.relative_to(output).parts)):
                archive.write(path, str(path.relative_to(output)))
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Ergebnis-ZIP ist beschädigt.')
    os.replace(temporary, target)
    print('Status:', status['state'], '| ZIP:', target, flush=True)
    try:
        from google.colab import files
        files.download(str(target))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(os.path.relpath(target, Path.cwd())))
        except Exception:
            print('ZIP aus diesem Verzeichnis herunterladen:', target)
if failure is not None:
    raise RuntimeError('Versuch angehalten; das Diagnose-ZIP enthält den tatsächlichen Fehler.') from None
print('Begrenzter explorativer H286-Gruppenversuch abgeschlossen. Inhaltliche Auswertung bleibt getrennt.')


Ergebnis-/Diagnose-ZIP herunterladen; vollständige Texte, Phasenstatus und Messwerte bleiben darin erhalten.
